# Mémoire M2 — firmware v10 : générateur corrigé, référence, ParEGO v2

**Ce notebook remplace `projets2/memoire_v8_phases1-3.ipynb` pour toutes les nouvelles mesures.**
L'ancien notebook et les campagnes v9 (`projets2/traces/`) ne sont plus modifiés : ils décrivent l'état « avant correction ».

| Élément | Emplacement |
|---|---|
| Firmware v10 | `firmware/mcu6/simpleserial-aes` (mcu5 = v9, inchangé) |
| Code Python | `chipwhisperer/projet` (copies des modules de projets2 + `banc_v10.py`) |
| Nouvelles traces | `/media/khadim/2A53DBCE3700DFDA/trace/v10/` |

**Ce qui change dans le firmware v10** (détails dans `noise_generator.h`) :
1. délais à **moyenne flottante** (Coron & Kizhvatov, CHES 2010), même coût moyen que les délais uniformes de v9 ;
2. **chemin rapide** des délais (surcoût des points d'injection réduit) ;
3. créneaux **factices sur le même chemin de code** que les réels (SubBytes et MixColumns) ;
4. générateur pseudo-aléatoire xoshiro128** **ensemencé par l'ADC** ;
5. commandes `E` (version, état de la graine) et `J` (réglage de la moyenne flottante) ; version « production ».
« Double rail » et « bruit arithmétique » ne sont plus utilisés (hors de l'espace de ParEGO).

**Règles**
- Un seul build v10 pour toutes les campagnes de ce notebook ; la référence sans bruit et `BASE_CYC` viennent de ce build.
- Métrique de ParEGO : **attaque CPA** (attaquant du mémoire, fenêtre (300, 24000)). Le CNN est réservé à l'évaluation finale.
- Ne lancer qu'un seul calcul lourd à la fois.

## 0. Configuration

In [ ]:
import os, sys, json, importlib
import numpy as np
import matplotlib.pyplot as plt

PROJET = "/home/khadim/chipwhisperer/projet"
os.chdir(PROJET)
for p in (PROJET, "/home/khadim/chipwhisperer/monitor"):
    if p not in sys.path:
        sys.path.insert(0, p)

import acceleration; acceleration.install()          # réalignement rapide (résultats identiques)
import banc_v10 as bv, sca_capture as sc, sca_metrics as sm, verify_n as vn, noise_opt as no, parego_v2 as pv
for m in (bv, sm, vn, no, pv):
    importlib.reload(m)

print("espace libre sur le disque de traces : %.0f Go" % bv.verifier_disque())
print("traces v10 :", bv.TRACE_ROOT)

## 1. Compilation et programmation du firmware v10
La compilation produit `simpleserial-aes-CWLITEARM.hex` dans `mcu6`. Pour la version « production » (sans commandes de banc),
ajouter `EXTRA_OPTS=NOISE_PRODUCTION` — **pas pour les mesures de ce notebook**.

In [ ]:
%%bash
cd ~/chipwhisperer/firmware/mcu6/simpleserial-aes/
make clean > /dev/null; rm -rf objdir-CWLITEARM simpleserial-aes-CWLITEARM.*
make PLATFORM='CWLITEARM' CRYPTO_TARGET=TINYAES128C SS_VER=SS_VER_1_1 2>&1 | tail -6

In [ ]:
nt = bv.connecter(programmer=True)
INFO = bv.info_firmware(nt)          # doit afficher : firmware v10 | graine ADC : oui

## 2. Tests fonctionnels
Vecteur de test FIPS-197 avec chaque générateur actif, puis vérification que la moyenne flottante augmente bien
la dispersion de la durée **à coût moyen égal** (même texte clair répété : seule la variabilité du bruit apparaît).

In [ ]:
import monitor_collect as mc
PT0 = np.frombuffer(bytes.fromhex("6bc1bee22e409f96e93d7e117393172a"), np.uint8)
CT0 = "3ad77bb40d7a3660a89ecaf32466ef97"

DELAIS_AVANT = lambda r: no.Config(((1, 1), (4, 1)), ((1, int(r)),), 0x403)          # délais zones 1 + 4
MELANGE      = no.Config(((1, 3), (3, 3), (4, 3)), ((3, 0),), 0x403)                    # mélange seul
TRES_FORT    = no.Config.from_dict(no.load_levels("levels_3niv.json")[2]["config"])    # niveau Très fort v9

tests = {"sans bruit": None, "délais avant (536)": DELAIS_AVANT(536), "mélange": MELANGE, "Très fort": TRES_FORT}
for nom, cfg in tests.items():
    mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
    bv.setup_v10(cfg)(nt)
    ok = all(bytes(sc.capture_one(nt, PT0)[1]).hex() == CT0 for _ in range(20))
    print("%-22s chiffré correct sur 20 essais : %s" % (nom, "oui" if ok else "NON"))

for s in (0, 2):
    t = bv.mesurer_durees(nt, DELAIS_AVANT(536), n=200, fm_shift=s, pt=PT0)
    print("fm_shift = %d (%s) : durée %.0f ± %.0f cycles" % (s, "uniforme v9" if s == 0 else "moyenne flottante",
                                                             t.mean(), t.std()))

## 3. Référence sans bruit v10 et `BASE_CYC`
Le chemin d'exécution a changé (table de pointeurs dans SubBytes) : **toutes** les comparaisons v10 se font par rapport à cette référence.
Graine 0 : mêmes textes clairs que `projets2/traces/comp20k`.

In [ ]:
P_REF = bv.capturer(nt, "sans_bruit_v10", 20000, cfg=None)
BASE_CYC = float(np.load(P_REF + "/trig.npy").mean())
json.dump(dict(BASE_CYC=BASE_CYC, ref=P_REF, firmware=INFO), open(os.path.join(bv.TRACE_ROOT, "base_v10.json"), "w"), indent=1)
print("BASE_CYC (v10) = %.0f cycles" % BASE_CYC)
bv.verifier_scope(nt, P_REF + "/meta.json")

In [ ]:
# ==== Affichage des 5 campagnes v10 (section 5) — hors ligne, carte non nécessaire ====
import os, json
import numpy as np
import matplotlib.pyplot as plt

TR = bv.TRACE_ROOT
B  = json.load(open(os.path.join(TR, "base_v10.json")))
CAMP = os.path.join(TR, "campagnes")
CAMPAGNES = {
    "sans protection":              B["ref"],
    "délais avant, moyenne flott.": CAMP + "/delais_avant_fm_20k",
    "délais avant, uniformes":      CAMP + "/delais_avant_uniforme_20k",
    "mélange seul":                 CAMP + "/melange_20k",
    "niveau Très fort":             CAMP + "/tres_fort_20k",
}
COUL = ["0.4", "#1f5fa8", "#b5651d", "#009E73", "#D55E00"]

def lire(p):
    X = np.load(p + "/traces.npy", mmap_mode="r")      # (N, S) int16, lu à la demande
    return X, np.load(p + "/trig.npy"), json.load(open(p + "/meta.json"))

N_SHOW, N_STAT = 8, 2000        # traces superposées / traces pour moyenne et σ
ZOOM = (300, 6000)              # zone où se concentre la fuite (cf. figure SNR 16.1)
ECH = 1024.0                    # même échelle que la cellule 16.2 (y / 1024)

# --- Figure 1 : traces individuelles, trace entière | zoom -------------------------------
fig, axs = plt.subplots(len(CAMPAGNES), 2, figsize=(15, 2.3 * len(CAMPAGNES)), sharex="col")
for i, (nom, p) in enumerate(CAMPAGNES.items()):
    X, trig, meta = lire(p)
    idx = np.linspace(0, len(X) - 1, N_SHOW, dtype=int)
    for k in idx:
        y = np.asarray(X[k], dtype=np.float32) / ECH
        axs[i, 0].plot(y, lw=0.3, alpha=0.7)
        axs[i, 1].plot(np.arange(*ZOOM), y[ZOOM[0]:ZOOM[1]], lw=0.4, alpha=0.7)
    axs[i, 0].set_ylabel(nom, fontsize=7)
    axs[i, 0].set_title("%d traces × %d échantillons | durée moy. %.0f cycles" % (len(X), X.shape[1], trig.mean()),
                        loc="left", fontsize=7)
axs[0, 1].set_title("zoom %d–%d" % ZOOM, loc="left", fontsize=7)
axs[-1, 0].set_xlabel("échantillon"); axs[-1, 1].set_xlabel("échantillon")
fig.tight_layout(); plt.show()

# --- Figure 2 : moyenne, écart-type, histogramme des durées ------------------------------
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for (nom, p), c in zip(CAMPAGNES.items(), COUL):
    X, trig, meta = lire(p)
    sub = np.asarray(X[:N_STAT, ZOOM[0]:ZOOM[1]], dtype=np.float32) / ECH
    axs[0].plot(np.arange(*ZOOM), sub.mean(0), lw=0.6, color=c, label=nom)
    axs[1].plot(np.arange(*ZOOM), sub.std(0),  lw=0.6, color=c, label=nom)
    axs[2].hist(trig, bins=80, histtype="step", lw=1.2, color=c, label="%s (σ = %.0f)" % (nom, trig.std()))
axs[0].set(title="Trace moyenne (%d traces)" % N_STAT, xlabel="échantillon")
axs[1].set(title="Écart-type par échantillon", xlabel="échantillon")
axs[2].set(title="Durée d'un chiffrement (trig)", xlabel="cycles"); axs[2].set_yscale("log")
axs[0].legend(fontsize=6); axs[2].legend(fontsize=6)
fig.tight_layout(); plt.show()

## 4. Délais avant AddRoundKey(0) à moyenne flottante : calibration et vérification
On recherche l'intensité qui donne le même surcoût que `delais_avant_20k` (v9, +42,6 %), puis on vérifie sur 2000 traces que
l'AddRoundKey(0) ne fuit plus à instant fixe (0 octet trouvé attendu).

In [ ]:
R_DELAIS, SUR = bv.calibrer(nt, DELAIS_AVANT, cible_pct=42.6, base_cyc=BASE_CYC, bornes=(100, 1500))
P_TEST = bv.capturer(nt, "delais_avant_fm_test2k", 2000, cfg=DELAIS_AVANT(R_DELAIS), sous_dossier="verification")
bv.rapport_durees(P_TEST, BASE_CYC)
v = bv.verif_ark0(P_TEST, P_REF, n=2000)

## 5. Campagnes de référence v10 (20 000 traces, clé fixe)
Quatre conditions au même protocole. `delais_avant_uniforme_v10` (fm_shift = 0) isole l'effet de la moyenne flottante.
Durée : ~15 min de capture par campagne.

In [ ]:
CAMPAGNES_V10 = {
    "sans protection":              P_REF,
    "délais avant, moyenne flott.": bv.capturer(nt, "delais_avant_fm_20k", 20000, cfg=DELAIS_AVANT(R_DELAIS)),
    "délais avant, uniformes":      bv.capturer(nt, "delais_avant_uniforme_20k", 20000, cfg=DELAIS_AVANT(R_DELAIS), fm_shift=0),
    "mélange seul":                 bv.capturer(nt, "melange_20k", 20000, cfg=MELANGE),
    "niveau Très fort":             bv.capturer(nt, "tres_fort_20k", 20000, cfg=TRES_FORT),
}
for nom, p in CAMPAGNES_V10.items():
    bv.rapport_durees(p, BASE_CYC)

## 6. Attaque directe des campagnes v10 (attaquant du mémoire, fenêtre (300, 24000))
Calcul seulement (la carte n'est pas nécessaire). Environ 20 à 40 min par campagne ; reprise automatique
(résultats dans `<campagne>_resultat_w300.json`).

In [ ]:
B = json.load(open(os.path.join(bv.TRACE_ROOT, "base_v10.json")))
C = json.load(open(os.path.join(bv.TRACE_ROOT, "calibration_v10.json")))
P_REF, BASE_CYC, R_DELAIS = B["ref"], C["BASE_CYC"], C["R_DELAIS"]
DELAIS_AVANT = lambda r: no.Config(((1, 1), (4, 1)), ((1, int(r)),), 0x403)
MELANGE      = no.Config(((1, 3), (3, 3), (4, 3)), ((3, 0),), 0x403)
TRES_FORT    = no.Config.from_dict(no.load_levels("levels_3niv.json")[2]["config"])
CAMP = os.path.join(bv.TRACE_ROOT, "campagnes")
CAMPAGNES_V10 = {
    "sans protection":              P_REF,
    "délais avant, moyenne flott.": CAMP + "/delais_avant_fm_20k",
    "délais avant, uniformes":      CAMP + "/delais_avant_uniforme_20k",
    "mélange seul":                 CAMP + "/melange_20k",
    "niveau Très fort":             CAMP + "/tres_fort_20k",
}
print("BASE_CYC = %.0f | R_DELAIS = %d" % (BASE_CYC, R_DELAIS))

In [ ]:
CLEAN = sm.clean_reference(P_REF, tuple(range(16)), window=bv.WIN)
RES_V10 = {nom: bv.evaluer(p, CLEAN) for nom, p in CAMPAGNES_V10.items()}
TABLEAU_V10 = bv.tableau(RES_V10, BASE_CYC)
json.dump(TABLEAU_V10, open(os.path.join(bv.TRACE_ROOT, "tableau_v10.json"), "w"), indent=1)

In [ ]:
# ---- Figure F2 : section 6 avec les surcoûts recalculés (réf. masque 0x000) ----
RES_V10 = {nom: json.load(open(p.rstrip("/") + "_resultat_w300.json")) for nom, p in CAMPAGNES_V10.items()}
NOMS6 = {"sans protection": "générateur inactif (masque 0x7FF)"}
print("%-36s %8s %9s %9s %10s" % ("condition", "cycles", "ancien", "réf 0x000", "X médian"))
for nom, r in RES_V10.items():
    print("%-36s %8.0f %+8.1f%% %+8.1f%% %10s" % (NOMS6.get(nom, nom), r["trig_mean"], 100 * (r["trig_mean"] / BASE_CYC - 1),
          surcout0(r["trig_mean"]), r["X_median"] or "> %d" % r["ns"][-1]))
fig, ax = plt.subplots(figsize=(8.0, 3.8))
for nom, r in RES_V10.items():
    ax.plot(r["ns"], r["effort_med"], marker="o", ms=3,
            label="%s (%+.0f %%)" % (NOMS6.get(nom, nom), surcout0(r["trig_mean"])))
ax.axhline(32, ls=":", c="0.4"); ax.text(22, 27, "seuil : 32 bits restants", fontsize=7, color="0.35")
ax.set_xscale("log")
ax.set_xlabel("traces dont dispose l'attaquant"); ax.set_ylabel("effort restant médian (bits)")
ax.set_title("Campagnes v10 (20 000 traces) ; surcoût par rapport au générateur désactivé", loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5)); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "effort_v10.pdf")); plt.show()

In [ ]:
# ---- Cellule F0 : données communes aux figures (sans carte) ----
from scipy.stats import kendalltau
DM = {int(k, 16): v for k, v in json.load(open(os.path.join(bv.TRACE_ROOT, "durees_masque_v10.json"))).items()}
REF0 = DM[0x000]                                  # AES, générateur désactivé (masque 0x000) : référence des surcoûts
BASE_PE = 45045.0                                 # base utilisée par ParEGO v2 pour overhead_pct (masque 0x7FF)
OPT11 = os.path.join(bv.TRACE_ROOT, "parego", "opt11")
H_PE  = json.load(open(os.path.join(OPT11, "history.json")))        # 1re mesure de chaque config
H_MOY = json.load(open(os.path.join(OPT11, "history_moy.json")))    # moyenne avec la répétition (A3)
H_REP = json.load(open(os.path.join(OPT11, "history_rep.json")))    # répétitions A3 (graine 43)
cycles = lambda h: BASE_PE * (1 + h["overhead_pct"] / 100)
surcout0 = lambda c: 100 * (c / REF0 - 1)
r10 = lambda h: bool(h["config"]["mask"] & 0x400)                   # la config protège-t-elle la ronde 10 ?
def front_pe(hist):
    return [hist[i] for i in sm.pareto_front([cycles(h) for h in hist], [h["log10_n"] for h in hist])]
print("référence des surcoûts : %.0f cycles | %d configs ParEGO dont %d protègent la ronde 10"
      % (REF0, len(H_MOY), sum(map(r10, H_MOY))))

## 7. ParEGO v2 (métrique CPA)
- **A1** (facultatif, sans carte) : ancien historique `projets2/traces/opt10` recalculé avec la fenêtre (300, 24000) — montre
  comment l'ancien front change quand on corrige seulement la métrique.
- **A2** (carte) : ParEGO sur le firmware v10, espace élargi (masques 0x003/0x007/0x403/0x7FF, délai 536), points de départ =
  niveaux actuels + délais avant. 6000 traces par point ; reprise automatique.
- **A3** (carte) : seconde campagne des points du front (autres textes clairs), puis moyenne.

In [ ]:
# A1 — sans carte (une nuit). Référence propre v9, car l'historique a été mesuré sur le firmware v9.
CLEAN_V9 = pv.make_clean(bv.ANCIEN + "/traces/v9_baseline")
HIST_V2 = pv.rescore_history(bv.ANCIEN + "/traces/opt10/history.json", CLEAN_V9, 42000.0,
                             out_path=os.path.join(bv.dossier("parego"), "opt10_history_win300.json"),
                             base_dir=bv.ANCIEN)
CMP12 = pv.compare_v1_v2(HIST_V2)

In [ ]:
# A2 — carte branchée, firmware v10
OPT = bv.dossier("parego", "opt11")
CLEAN_PE = pv.make_clean(P_REF)
seeds = pv.seed_configs("levels_3niv.json", rescored=HIST_V2 if "HIST_V2" in globals() else None)
seeds = [c if c.name != pv.CFG_DELAIS_AVANT.name else DELAIS_AVANT(R_DELAIS) for c in seeds]
opt = pv.run_parego_v2(nt, bv.KEY, CLEAN_PE, BASE_CYC, seeds=seeds, history_path=os.path.join(OPT, "history.json"),
                       n_init=6, n_iter=25, max_overhead=50, n_traces=6000)

In [ ]:
# A3 — répétition des points du front
HIST_MOY = pv.replicate_front(nt, bv.KEY, CLEAN_PE, BASE_CYC, history_path=os.path.join(OPT, "history.json"), n_traces=6000)
for h in pv.front(HIST_MOY):
    print("%-80s %+6.1f %%  log10N = %.2f ± %.2f" % (h["config"]["name"][:80], h["overhead_pct"], h["log10_n"],
                                                   h.get("log10_n_sd", float("nan"))))

# front ParEGO v2

In [ ]:
# ---- Figure F3 : front ParEGO v2 (surcoût réf. 0x000) ----
P10, P03 = [h for h in H_MOY if r10(h)], [h for h in H_MOY if not r10(h)]
F10 = front_pe(P10)
NOMMES = {
    "permutation@subbytes_pre:4+jitter@mixcolumns:100+permutation@mixcolumns:4+permutation@addroundkey_pre:4|m403": ("Très fort", (6, -12)),
    "jitter@subbytes_pre:712+jitter@addroundkey_pre:712|m403": ("délais avant (712)", (6, 4)),
    "permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403": ("mélange + délai avant AddRoundKey", (6, -14)),
    "permutation@subbytes_pre:0+jitter@mixcolumns:25+permutation@mixcolumns:0+permutation@addroundkey_pre:0|m403": ("Fort", (6, -10)),
}
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.scatter([surcout0(cycles(h)) for h in P03], [h["log10_n"] for h in P03], s=14, facecolors="none", edgecolors="0.65",
           lw=0.8, label="ronde 10 non protégée (0x003 / 0x007) : non retenue")
ax.scatter([surcout0(cycles(h)) for h in P10], [h["log10_n"] for h in P10], s=18, c="#1f5fa8",
           label="ronde 10 protégée (0x403 / 0x7FF)")
ax.step([surcout0(cycles(h)) for h in F10], [h["log10_n"] for h in F10], where="post", c="#1f5fa8", lw=1)
plafond = max(h["log10_n"] for h in H_MOY)
ax.axhline(plafond, ls=":", c="0.4", lw=0.8); ax.text(95, plafond + 0.06, "plafond de la mesure à 6000 traces", fontsize=6, color="0.35")
for h in P10:
    if h["config"]["name"] in NOMMES:
        lab, off = NOMMES[h["config"]["name"]]
        ax.annotate(lab, (surcout0(cycles(h)), h["log10_n"]), xytext=off, textcoords="offset points", fontsize=6.5, color="#1f5fa8")
ax.set_xlabel("surcoût en temps par rapport au générateur désactivé (masque 0x000, %.0f cycles) [%%]" % REF0, fontsize=8)
ax.set_ylabel("log₁₀ N estimé (octet le plus faible, CPA)", fontsize=8)
ax.set_title("ParEGO v2 (firmware v10) : front des configurations qui protègent la ronde 10", loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=6.5, loc="center right"); ax.set_ylim(0.85, 4.2); ax.set_xlim(left=-2)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "front_parego_v10.pdf")); plt.show()
print("front (ronde 10 protégée) :")
for h in F10:
    print("  %6.0f cycles  %+5.1f %%  log10N = %.2f%s  %s" % (cycles(h), surcout0(cycles(h)), h["log10_n"],
          " (plafond)" if h.get("censored") else "", h["config"]["name"]))

In [ ]:
# ---- Figure F4 : reproductibilité de la mesure rapide (A3, graine 42 puis 43) ----
P1 = {h["config"]["name"]: h for h in H_PE}
paires = [(n, P1[n]["log10_n"], r["log10_n"]) for n, r in H_REP.items() if n in P1]
a, b = np.array([p[1] for p in paires]), np.array([p[2] for p in paires])
print("%d configs répétées | écart |Δ log10N| : médiane %.2f, max %.2f" % (len(paires), np.median(abs(a - b)), abs(a - b).max()))
fig, ax = plt.subplots(figsize=(3.8, 3.6))
lim = (0.9, plafond + 0.15)
ax.fill_between(lim, [lim[0] - 0.1, lim[1] - 0.1], [lim[0] + 0.1, lim[1] + 0.1], color="0.9", label="± 0,1 décade")
ax.plot(lim, lim, c="0.5", lw=0.8)
ax.scatter(a, b, s=20, c=["#1f5fa8" if n.endswith(("m403", "m7FF")) else "white" for n, _, _ in paires],
           edgecolors="#1f5fa8", zorder=3)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("log₁₀ N, 1re mesure (graine 42)", fontsize=8); ax.set_ylabel("log₁₀ N, répétition (graine 43)", fontsize=8)
ax.set_title("Mesure rapide reproductible (%d points du front)" % len(paires), loc="left", fontsize=9)
from matplotlib.lines import Line2D
ax.legend(handles=[ax.collections[0], Line2D([], [], ls="", marker="o", mfc="#1f5fa8", mec="#1f5fa8", label="ronde 10 protégée"),
                   Line2D([], [], ls="", marker="o", mfc="white", mec="#1f5fa8", label="masque 0x003 / 0x007")],
          frameon=False, fontsize=7, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "reproductibilite_a3.pdf")); plt.show()

In [ ]:
# ---- Figure F5 : A1, ancien historique (opt10, v9) recalculé avec la fenêtre (300, 24000) ----
A1 = json.load(open(os.path.join(bv.TRACE_ROOT, "parego", "opt10_history_win300.json")))
v1, v2 = np.array([h["log10_n_v1"] for h in A1]), np.array([h["log10_n"] for h in A1])
tau = kendalltau(v1, v2)[0]
print("%d configs | tau de Kendall = %.2f | variation médiane = %+.2f décade" % (len(A1), tau, np.median(v2 - v1)))
fig, ax = plt.subplots(figsize=(3.8, 3.6))
lim = (min(v1.min(), v2.min()) - 0.1, max(v1.max(), v2.max()) + 0.15)
ax.plot(lim, lim, c="0.5", lw=0.8)
ax.scatter(v1, v2, s=18, c="#1f5fa8")
ax.text(0.04, 0.92, "τ de Kendall = %.2f" % tau, transform=ax.transAxes, fontsize=8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("log₁₀ N, fenêtre (1500, 24000) [ParEGO v1]", fontsize=8)
ax.set_ylabel("log₁₀ N, fenêtre (300, 24000)", fontsize=8)
ax.set_title("La fenêtre seule ne change pas l'ancien front", loc="left", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "a1_fenetre.pdf")); plt.show()

## 8. Validation du front (A4) — 20 000 traces par point
Capture dans le notebook, attaque **dans un terminal** (long). Deux campagnes par configuration retenue (graines 0 et 1) :
la variabilité entre campagnes observée en v9 atteint un facteur ~2,5 sur X pour une même configuration.

**Déjà fait (cellule suivante, exécutée le 30/10)** : 8 campagnes graine 0 dans `validation/`, prises sur le front global.
7 d'entre elles ont le masque 0x003 : elles ne sont **pas** retenues pour les niveaux, car la capture
(24 400 échantillons) ne voit pas la ronde 10 ; seul le candidat
`permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403` est utile.
La cellule est désactivée (`DEJA_FAIT = True`) pour ne pas recapturer.

In [ ]:
DEJA_FAIT = True          # cellule exécutée le 30/10 (8 campagnes graine 0) — ne pas relancer
if not DEJA_FAIT:
    VAL = bv.dossier("validation")
    a_valider = [no.Config.from_dict(h["config"]) for h in pv.front(HIST_MOY) if h["overhead_pct"] <= 50]
    IDX = pv.capture_validation(nt, bv.KEY, a_valider, outdir=VAL, n_traces=20000)
    print(len(IDX), "campagnes de validation dans", VAL)
    print("Commande à lancer dans un terminal :")
    print("  cd %s && nohup python valide_front_v2.py %s --clean %s --base-cyc %.0f --nproc 10 --out %s > %s 2>&1 &"
          % (bv.PROJET, VAL, P_REF, BASE_CYC, os.path.join(VAL, "resultats"), os.path.join(VAL, "log_validation.txt")))

### 8.1 Durée de référence dans la même session (carte branchée, ~1 min)
ParEGO a mesuré des configurations bruitées **plus rapides** que la référence sans bruit du 28/10
(ex. `jitter@subbytes_pre:10|m003` : 41 160 cycles contre `BASE_CYC` = 45 045). Cette cellule mesure, dans la même
session, la référence sans bruit (toutes rondes), la configuration « aucun générateur » avec le masque 0x403, Normal et
Très fort. Si le kernel a redémarré : exécuter les cellules 0 (imports) et la cellule de reprise de la section 6, puis
`nt = bv.connecter(programmer=False)`.

In [ ]:
TESTS = {"sans bruit (make_setup par défaut)": None,
         "aucun générateur, masque 0x403":      no.Config((), (), 0x403),
         "Normal (jitter@subbytes_pre:10|m403)": no.Config(((1, 1),), ((1, 10),), 0x403),
         "Très fort":                            TRES_FORT}
DUREES_SESSION = {}
for nom, cfg in TESTS.items():
    t = bv.mesurer_durees(nt, cfg, n=200)
    DUREES_SESSION[nom] = (float(t.mean()), float(t.std()))
    print("%-38s %7.0f ± %4.0f cycles" % (nom, t.mean(), t.std()))
print("référence enregistrée (28/10, trig.npy de sans_bruit_v10) : %.0f cycles" % BASE_CYC)

In [ ]:
DUREES_MASQUE = {}
for m in (0x000, 0x001, 0x003, 0x007, 0x403, 0x7FF):
    t = bv.mesurer_durees(nt, no.Config((), (), m), n=50)
    DUREES_MASQUE[m] = float(t.mean())
    print("aucun générateur, masque 0x%03X (%2d rondes) : %7.0f ± %3.0f cycles"
          % (m, bin(m).count("1"), t.mean(), t.std()))
json.dump({"%03X" % m: v for m, v in DUREES_MASQUE.items()},
          open(os.path.join(bv.TRACE_ROOT, "durees_masque_v10.json"), "w"), indent=1)

In [ ]:
# ---- Figure F1 : coût du générateur désactivé selon le masque de rondes ----
APPELS = {0: 18, 10: 38}                          # points d'insertion par ronde (aes.c) ; rondes 1-9 : 43
CPA = 10.8                                        # cycles par point d'insertion actif (ajusté ci-dessous)
def predit(m):
    return REF0 + CPA * sum(APPELS.get(r, 43) for r in range(11) if m >> r & 1)
cpa_fit = (DM[0x7FF] - REF0) / sum(APPELS.get(r, 43) for r in range(11))
print("cycles par point d'insertion actif (ajustement 0x000 -> 0x7FF) : %.2f" % cpa_fit)
masques = sorted(DM, key=lambda m: (bin(m).count("1"), m))
fig, ax = plt.subplots(figsize=(5.6, 3.2))
x = np.arange(len(masques))
ax.bar(x, [DM[m] - REF0 for m in masques], color="#1f5fa8", width=0.6, label="mesuré")
ax.plot(x, [predit(m) - REF0 for m in masques], "o", mfc="white", mec="k", ms=5, label="modèle : 10,8 cycles par point d'insertion")
for i, m in enumerate(masques):
    ax.text(i, DM[m] - REF0 + 80, "%+d" % (DM[m] - REF0), ha="center", fontsize=7)
ax.set_xticks(x, ["0x%03X\n(%d ronde%s)" % (m, bin(m).count("1"), "s" if bin(m).count("1") > 1 else "") for m in masques], fontsize=7)
ax.set_ylabel("cycles ajoutés au chiffrement\n(réf. masque 0x000 : %.0f cycles)" % REF0)
ax.set_title("Générateur désactivé : chaque ronde du masque coûte ses tests", loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "cout_masque_v10.pdf")); plt.show()

### 8.2 Captures de validation complémentaires (carte branchée, ~2 h 15)
Candidats retenus parmi les configurations qui protègent la ronde 10 (front ParEGO v2, `history_moy.json`) :

| Candidat | Durée ParEGO (cycles) | log₁₀ N proxy |
|---|---|---|
| mélange SubBytes/MixColumns/AddRoundKey + délai 536 avant AddRoundKey | 57 673 | ≥ 3,96 (plafond) |
| mélange MixColumns + délai 536 et mélange (4) avant AddRoundKey | 57 357 | 3,16 |
| Fort (v9) | 48 659 | 2,32 |
| délais 100 en début de ronde et avant SubBytes + mélange avant AddRoundKey | 47 666 | 2,31 |

Références : Très fort et délais avant (712). En graine 0, elles existent déjà (`campagnes/tres_fort_20k`,
`campagnes/delais_avant_fm_20k`, attaquées en section 6) : on les recapture seulement en graine 1.
Au total : 3 campagnes graine 0 + 6 campagnes graine 1 (≈ 0,9 Go chacune). Reprise automatique si interruption.

In [ ]:
bv.verifier_disque(min_go=12)
H_MOY = {h["config"]["name"]: h for h in json.load(open(os.path.join(bv.TRACE_ROOT, "parego/opt11/history_moy.json")))}
CANDIDATS = [
    "permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403",
    "permutation@mixcolumns:4+jitter@addroundkey_pre:536+permutation@addroundkey_pre:4|m403",
    "permutation@subbytes_pre:0+jitter@mixcolumns:25+permutation@mixcolumns:0+permutation@addroundkey_pre:0|m403",
    "jitter@round_start:100+jitter@subbytes_pre:100+permutation@addroundkey_pre:0|m403",
]
CAND = [no.Config.from_dict(H_MOY[n]["config"]) for n in CANDIDATS]
REFS = [TRES_FORT, DELAIS_AVANT(R_DELAIS)]
assert REFS[1].name == "jitter@subbytes_pre:712+jitter@addroundkey_pre:712|m403", REFS[1].name

def _deja(outdir):
    p = os.path.join(outdir, "index.json")
    idx = json.load(open(p)) if os.path.exists(p) else {}
    return {n for n, c in idx.items() if os.path.exists(os.path.join(c, "meta.json"))}

VAL0, VAL1 = bv.dossier("validation"), bv.dossier("validation", "s1")
a_faire0 = [c for c in CAND if c.name not in _deja(VAL0)]
a_faire1 = [c for c in CAND + REFS if c.name not in _deja(VAL1)]
print("à capturer : %d en graine 0, %d en graine 1" % (len(a_faire0), len(a_faire1)))
IDX0 = pv.capture_validation(nt, bv.KEY, a_faire0, outdir=VAL0, n_traces=20000, seed=0)
IDX1 = pv.capture_validation(nt, bv.KEY, a_faire1, outdir=VAL1, n_traces=20000, seed=1)
for nom, p in sorted(IDX1.items()):
    print("%-100s %.0f cycles" % (nom, np.load(p + "/trig.npy").mean()))

### 8.3 Attaque des campagnes de validation — **dans un terminal** (une nuit)
Même attaquant qu'en section 6. Le nombre de processus est limité automatiquement (`bv.nproc_sur()`, ~3 Go
par processus) : **ne pas** utiliser `--nproc 10` (la commande affichée par l'ancienne cellule 8 provoque un manque
de mémoire). Reprise automatique : relancer la même commande après une interruption.

```
cd ~/chipwhisperer/projet
nohup python valider_v10.py > /media/khadim/2A53DBCE3700DFDA/trace/v10/validation/log_validation.txt 2>&1 &
tail -f /media/khadim/2A53DBCE3700DFDA/trace/v10/validation/log_validation.txt
```
Seules les configurations qui protègent la ronde 10 sont attaquées (11 campagnes, dont 2 déjà calculées) ;
`TOUT=1 nohup python valider_v10.py …` ajoute les 7 campagnes 0x003.

### 8.4 Lecture des résultats de la validation (sans carte)

In [ ]:
# ---- Cellule 8.4 : résultats de la validation A4 (tableau + figure) ----
RV = json.load(open(os.path.join(bv.TRACE_ROOT, "validation", "resultats_validation.json")))
REF0 = json.load(open(os.path.join(bv.TRACE_ROOT, "durees_masque_v10.json")))["000"]
H_PROXY = {h["config"]["name"]: h for h in json.load(open(os.path.join(bv.TRACE_ROOT, "parego", "opt11", "history_moy.json")))}
LIB = {   # libellés lisibles
    "permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403": "mélange (3 zones) + délai avant AddRoundKey",
    "permutation@mixcolumns:4+jitter@addroundkey_pre:536+permutation@addroundkey_pre:4|m403": "mélange MixColumns + délai et mélange avant AddRoundKey",
    "permutation@subbytes_pre:0+jitter@mixcolumns:25+permutation@mixcolumns:0+permutation@addroundkey_pre:0|m403": "Fort (v9)",
    "jitter@round_start:100+jitter@subbytes_pre:100+permutation@addroundkey_pre:0|m403": "délais 100 + mélange avant AddRoundKey",
    "permutation@subbytes_pre:4+jitter@mixcolumns:100+permutation@mixcolumns:4+permutation@addroundkey_pre:4|m403": "Très fort (v9)",
    "jitter@subbytes_pre:712+jitter@addroundkey_pre:712|m403": "délais avant (712)",
}
noms = sorted(LIB, key=lambda n: np.mean([RV[s][n]["trig_mean"] for s in RV if n in RV[s]]))
print("%-52s %7s %6s | %-6s | %8s %8s | %13s | %11s" % ("configuration", "cycles", "surc.", "proxy", "X s0", "X s1",
      "bits à 8000", "octets trouvés"))
for n in noms:
    r0, r1 = RV["s0"][n], RV["s1"][n]
    cyc = (r0["trig_mean"] + r1["trig_mean"]) / 2
    X = lambda r: str(r["X_median"]) if r["X_median"] else "> %d" % r["ns"][-1]
    print("%-52s %7.0f %+5.1f%% | %6.2f | %8s %8s | %5.1f / %-5.1f | %4.1f / %-4.1f" % (LIB[n], cyc, 100 * (cyc / REF0 - 1),
          H_PROXY[n]["log10_n"], X(r0), X(r1), r0["effort_med"][-1], r1["effort_med"][-1], r0["octets"][-1], r1["octets"][-1]))
fig, ax = plt.subplots(figsize=(7.5, 5.2))
cmap = plt.get_cmap("viridis")
for i, n in enumerate(noms):
    col = cmap(i / (len(noms) - 1) * 0.9)
    cyc = np.mean([RV[s][n]["trig_mean"] for s in ("s0", "s1")])
    for s, ls in (("s0", "-"), ("s1", "--")):
        r = RV[s][n]
        ax.plot(r["ns"], r["effort_med"], ls=ls, marker="o", ms=2.5, color=col,
                label="%s (%+.0f %%)" % (LIB[n], 100 * (cyc / REF0 - 1)) if s == "s0" else None)
ax.axhline(32, ls=":", c="0.4"); ax.text(8000, 34, "seuil : 32 bits restants (clé retrouvée en dessous)", fontsize=7, color="0.35", ha="right")
ax.set_xscale("log"); ax.set_ylim(-3, 110)
ax.set_xlabel("traces dont dispose l'attaquant (meilleur des attaquants A1 à A4)")
ax.set_ylabel("effort restant médian (bits)")
ax.set_title("Validation A4 : 20 000 traces, deux campagnes par configuration\n(trait plein : graine 0, tirets : graine 1)",
             loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
          title="(surcoût par rapport au générateur désactivé)", title_fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "validation_a4.pdf")); plt.show()

# 8.5-a — captures (carte branchée, environ 2 h 10)

In [ ]:
# ==== Cellule 8.5-a : captures pour figer les niveaux (carte branchée, ~2 h 10) ====
assert bv.empreinte_firmware() == json.load(open(os.path.join(bv.TRACE_ROOT, "calibration_v10.json")))["build"], \
    "le firmware sur disque n'est pas le build v10 de référence"
bv.verifier_disque(min_go=12)

# Niveau 3 validé (A4) : mélange SubBytes + MixColumns + AddRoundKey, délai avant AddRoundKey, rondes 0, 1, 10
CAND3 = no.Config.from_dict({"comps": [[1, 3], [3, 3], [4, 1], [4, 3]], "raw": [[1, 536], [3, 0]], "mask": 0x403})
famille = lambda d: no.Config(CAND3.comps, tuple((g, d if g == 1 else v) for g, v in CAND3.raw), CAND3.mask)
FAMILLE = [famille(d) for d in (100, 200, 400)]          # option B : intensité du délai avant AddRoundKey
for c in FAMILLE:
    print(c.name)

def _deja(outdir):
    p = os.path.join(outdir, "index.json")
    idx = json.load(open(p)) if os.path.exists(p) else {}
    return {n for n, c in idx.items() if os.path.exists(os.path.join(c, "meta.json"))}

# Option B : 3 intensités x 2 graines (20 000 traces chacune, ~15 min par campagne)
VAL0, VAL1 = bv.dossier("validation"), bv.dossier("validation", "s1")
pv.capture_validation(nt, bv.KEY, [c for c in FAMILLE if c.name not in _deja(VAL0)], outdir=VAL0, n_traces=20000, seed=0)
pv.capture_validation(nt, bv.KEY, [c for c in FAMILLE if c.name not in _deja(VAL1)], outdir=VAL1, n_traces=20000, seed=1)

# Option C : grande campagne du niveau 3 (50 000 traces, graine 2, ~35 min, ~2,5 Go)
P_GRANDE = bv.capturer(nt, "candidat_50k", 50000, cfg=CAND3, sous_dossier="validation/grande", seed=2)
for p in [os.path.join(VAL0, pv._safe(c.name)) for c in FAMILLE] + [os.path.join(VAL1, pv._safe(c.name)) for c in FAMILLE] + [P_GRANDE]:
    t = np.load(os.path.join(p, "trig.npy"))
    print("%-110s %6d traces | %7.0f cycles" % (p.split("/v10/")[1], len(t), t.mean()))

# 8.5-b — attaques de nuit (sans carte)

In [ ]:
# ==== Cellule 8.5-b : lancement des attaques en arrière-plan (sans carte, une nuit) ====
import subprocess
VALD = os.path.join(bv.TRACE_ROOT, "validation")
CMD = ("%s valider_v10.py >> %s/log_validation.txt 2>&1; NPROC=3 %s grande_v10.py > %s/grande/log_grande.txt 2>&1"
       % (sys.executable, VALD, sys.executable, VALD))
proc_nuit = subprocess.Popen(["sh", "-c", CMD], cwd=bv.PROJET, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
                             start_new_session=True)               # indépendant du kernel
print("attaques lancées, PID", proc_nuit.pid)

In [ ]:
# ==== Cellule 8.5-c : suivi des attaques ====
VALD = os.path.join(bv.TRACE_ROOT, "validation")
for f in ("log_validation.txt", "grande/log_grande.txt"):
    p = os.path.join(VALD, f)
    print("=====", f); print(open(p).read()[-1200:] if os.path.exists(p) else "(pas encore démarré)")

### 8.6 Niveaux : N à 10 % de réussite de l'attaquant, enveloppe convexe (sans carte)

In [ ]:
# ==== Cellule 8.6 : famille de niveaux, enveloppe convexe et choix des niveaux (sans carte) ====
T10 = bv.TRACE_ROOT
REF0 = json.load(open(os.path.join(T10, "durees_masque_v10.json")))["000"]         # générateur désactivé
RV = json.load(open(os.path.join(T10, "validation", "resultats_validation.json")))
RG = json.load(open(os.path.join(T10, "validation", "grande", "candidat_50k_resultat_w300.json")))
RM = json.load(open(os.path.join(T10, "campagnes", "melange_20k_resultat_w300.json")))   # délai 0 = mélange seul
BASE_N = 20                                                                      # générateur désactivé : X = 20
fam_nom = lambda d: ("permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:%d+"
                     "permutation@addroundkey_pre:0|m403" % d)
FORT = "permutation@subbytes_pre:0+jitter@mixcolumns:25+permutation@mixcolumns:0+permutation@addroundkey_pre:0|m403"

def n_prudent(rs):
    """N prudent = plus petit n où 10 % des essais de l'attaquant laissent <= 32 bits (minimum sur les campagnes) ;
    si aucune campagne n'atteint ce point, borne inférieure (n max mesuré)."""
    n10 = lambda r: next((int(n) for n, e in zip(r["ns"], r["effort_p10"]) if e <= 32), None)
    vals = [n10(r) for r in rs]
    cass = [v for v in vals if v is not None]
    return (min(cass), False) if cass else (max(int(r["ns"][-1]) for r in rs), True)

POINTS = []          # (libellé, délai d, cycles, N prudent, borne ?, détail)
POINTS.append(("générateur désactivé", None, REF0, BASE_N, False, "réf."))
POINTS.append(("mélange seul (délai 0)", 0, RM["trig_mean"], *n_prudent([RM]), "1 campagne"))
rf = [RV[s][FORT] for s in ("s0", "s1")]
POINTS.append(("Fort (v9)", None, np.mean([r["trig_mean"] for r in rf]), *n_prudent(rf), "X %s / %s" % tuple(r["X_median"] for r in rf)))
for d in (100, 200, 400, 536):
    rs = [RV[s][fam_nom(d)] for s in ("s0", "s1")] + ([RG] if d == 536 else [])
    N, borne = n_prudent(rs)
    POINTS.append(("mélange + délai %d avant ARK" % d, d, np.mean([r["trig_mean"] for r in rs]), N, borne,
                   " / ".join(str(r["X_median"] or "> %d" % r["ns"][-1]) for r in rs)))
AUTRES = {   # autres configurations validées en A4 (hors famille) : elles entrent aussi dans l'enveloppe
    "Très fort (v9)": "permutation@subbytes_pre:4+jitter@mixcolumns:100+permutation@mixcolumns:4+permutation@addroundkey_pre:4|m403",
    "délais avant (712)": "jitter@subbytes_pre:712+jitter@addroundkey_pre:712|m403",
    "mélange MC + délai 536 + mélange ARK": "permutation@mixcolumns:4+jitter@addroundkey_pre:536+permutation@addroundkey_pre:4|m403",
    "délais 100 + mélange ARK": "jitter@round_start:100+jitter@subbytes_pre:100+permutation@addroundkey_pre:0|m403",
}
for lab, n in AUTRES.items():
    rs = [RV[s][n] for s in ("s0", "s1")]
    POINTS.append((lab, None, np.mean([r["trig_mean"] for r in rs]), *n_prudent(rs),
                   " / ".join(str(r["X_median"] or "> %d" % r["ns"][-1]) for r in rs)))
surc = lambda c: 100 * (c / REF0 - 1)

# enveloppe convexe inférieure de (surcoût, 1/N) : un mélange aléatoire de deux niveaux voisins à taux p
# a un coût et une fuite par chiffrement (1/N) linéaires en p -> seuls les points de l'enveloppe sont utiles
P = sorted(POINTS, key=lambda p: p[2])
xs, ys = np.array([surc(p[2]) for p in P]), np.array([1 / p[3] for p in P])
hull = [0]
for i in range(1, len(P)):
    while len(hull) >= 2:
        a, b = hull[-2], hull[-1]
        if (ys[b] - ys[a]) * (xs[i] - xs[a]) >= (ys[i] - ys[a]) * (xs[b] - xs[a]):   # b au-dessus du segment a-i
            hull.pop()
        else:
            break
    if ys[i] < ys[hull[-1]]:
        hull.append(i)
print("%-32s %8s %7s %9s %-24s %s" % ("point", "cycles", "surc.", "N (10 %)", "X médian des campagnes", "enveloppe"))
for i, p in enumerate(P):
    print("%-32s %8.0f %+6.1f%% %s%8d %-24s %s" % (p[0], p[2], surc(p[2]), "≥" if p[4] else " ", p[3], p[5],
                                                   "oui" if i in hull else "non (dominé)"))
NIVEAUX = {1: fam_nom(100), 2: fam_nom(200), 3: fam_nom(536)}      # une seule config, un seul bouton : le délai d

fig, ax1 = plt.subplots(figsize=(6.4, 3.8))
fam = [p for p in P if p[1] is not None]
d_ = [p[1] for p in fam]
ax1.plot(d_, [p[3] for p in fam], "-", c="#1f5fa8", label="N à 10 % de réussite (traces)")
for p in fam:      # plein : clé cassée ; creux + flèche : non cassée, N est une borne inférieure
    ax1.plot(p[1], p[3], "o", mfc="white" if p[4] else "#1f5fa8", mec="#1f5fa8", ms=6)
    if p[4]:
        ax1.annotate("", (p[1], p[3] * 1.9), (p[1], p[3] * 1.08), arrowprops=dict(arrowstyle="->", color="#1f5fa8"))
    ax1.annotate(("≥ " if p[4] else "") + "%d" % p[3], (p[1], p[3]), xytext=(-10, -14), textcoords="offset points",
                 fontsize=7, color="#1f5fa8", ha="right")
ax1.set_yscale("log"); ax1.axhline(1e4, ls=":", c="0.4"); ax1.text(5, 1.15e4, "cible N_vie = 10⁴", fontsize=7, color="0.35")
ax1.set_xlabel("intensité du délai avant AddRoundKey (cycles bruts, mélange actif)")
ax1.set_ylabel("N (10 % de réussite) : traces", color="#1f5fa8")
ax2 = ax1.twinx(); ax2.plot(d_, [surc(p[2]) for p in fam], "s--", c="0.45", ms=4, label="surcoût (%)")
ax1.set_ylim(50, 1e5)
fig.legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(0.12, 0.88))
ax2.set_ylabel("surcoût (%)", color="0.45")
ax1.set_title("Un seul réglage règle le compromis coût / sécurité (famille du niveau 3)", loc="left", fontsize=9)
ax1.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(T10, "famille_niveaux.pdf")); plt.show()

### 8.7 Plancher calibré selon la durée de vie visée N_vie (+ plancher embarqué dans le firmware v10.1)

In [ ]:
# ==== Cellule 8.7 : plancher calibré (coût minimal garanti selon la durée de vie visée N_vie) ====
# Budget de fuite sur la vie de la clé (sans changement de clé) : somme des n_i / N_i <= 1 / m.
# Avec un mélange constant entre deux niveaux voisins de l'enveloppe : fuite par chiffrement = p/N_a + (1-p)/N_b.
HP = [P[i] for i in hull]                        # points de l'enveloppe, coût croissant
hx, hy = np.array([surc(p[2]) for p in HP]), np.array([1 / p[3] for p in HP])
def plancher(n_vie, m):
    cible = 1 / (m * n_vie)                      # fuite moyenne autorisée par chiffrement
    if cible >= hy[0]:
        return 0.0, HP[0][0], HP[0][0], 1.0
    for a in range(len(HP) - 1):
        if hy[a] >= cible >= hy[a + 1]:
            p = (cible - hy[a + 1]) / (hy[a] - hy[a + 1])          # part du niveau a (le moins cher)
            return p * hx[a] + (1 - p) * hx[a + 1], HP[a][0], HP[a + 1][0], p
    return None, HP[-1][0], None, 0.0             # aucun niveau mesuré ne suffit
for m in (2, 4):
    print("marge m = %d" % m)
    for n_vie in (1e2, 1e3, 3e3, 1e4, 3e4):
        c, a, b, p = plancher(n_vie, m)
        print("  N_vie = %6d : %s" % (n_vie, "aucun niveau suffit (N du niveau 3 >= %d seulement)" % HP[-1][3] if c is None
              else "surcoût %5.1f %%  = %3.0f %% « %s » + %3.0f %% « %s »" % (c, 100 * p, a, 100 * (1 - p), b)))
# Plancher effectivement embarqué dans le firmware v10.1 : seulement les 3 niveaux de la famille (d = 100, 200, 536)
import politique_v10 as pol; importlib.reload(pol)
NIV_FW, _ = pol.charger_niveaux()
print("firmware v10.1 (niveaux d = 100 / 200 / 536, N à 10 %% : %s) :" % " / ".join(("≥" if v["borne"] else "") + str(v["N"]) for v in NIV_FW))
for m in (2, 4):
    r = pol.plancher(NIV_FW, 1e4, m)
    if r is None:
        print("  m = %d, N_vie = 10⁴ : aucun niveau ne suffit" % m)
    else:
        a, b, p = r
        print("  m = %d, N_vie = 10⁴ : %.0f %% niveau %d + %.0f %% niveau %d -> surcoût %.1f %%"
              % (m, 100 * p, a + 1, 100 * (1 - p), b + 1, p * NIV_FW[a]["surcout"] + (1 - p) * NIV_FW[b]["surcout"]))
fig, ax = plt.subplots(figsize=(6.0, 3.6))
nv = np.logspace(1.5, 4.2, 200)
for m, ls in ((2, "-"), (4, "--")):
    cs = [plancher(n, m)[0] for n in nv]
    ax.plot(nv, [np.nan if c is None else c for c in cs], ls, c="#1f5fa8", label="marge m = %d" % m)
ax.axvline(1e4, ls=":", c="0.4"); ax.text(9.0e3, 16, "N_vie = 10⁴", fontsize=7, color="0.35", ha="right")
ax.set_xscale("log"); ax.set_xlabel("durée de vie visée de la clé N_vie (chiffrements observables)")
ax.set_ylabel("surcoût minimal garanti (%)")
ax.set_title("Plancher calibré : coût de la garantie selon la durée de vie visée", loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(T10, "plancher_calibre.pdf")); plt.show()

### 8.8 Front de Pareto validé (attaque complète)

In [ ]:
# ==== Cellule 8.8 : front de Pareto VALIDÉ (attaque complète, 20 000 à 50 000 traces) ====
# Nécessite la cellule 8.6 (POINTS, hull, RV, REF0, surc, n_prudent, fam_nom).
TOUS = list(POINTS)
H = [P[i] for i in hull]
noms_h = {p[0] for p in H}
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.step([surc(p[2]) for p in H], [p[3] for p in H], where="post", c="#1f5fa8", lw=1, zorder=1)
for p in TOUS:
    sur, col = p[0] in noms_h, ("#1f5fa8" if p[0] in noms_h else "0.6")
    ax.plot(surc(p[2]), p[3], "o", ms=6 if sur else 5, mec=col, mfc="white" if p[4] else col, zorder=3)
    if p[4]:
        ax.annotate("", (surc(p[2]), p[3] * 1.8), (surc(p[2]), p[3] * 1.08), arrowprops=dict(arrowstyle="->", color=col))
NIV = {fam_nom(100): "niveau 1", fam_nom(200): "niveau 2", fam_nom(536): "niveau 3"}
ETIQ = {"mélange + délai 100 avant ARK": ("niveau 1 (d = 100)", (-8, 8), "right"),
        "mélange + délai 200 avant ARK": ("niveau 2 (d = 200)", (6, -12), "left"),
        "mélange + délai 536 avant ARK": ("niveau 3 (d = 536)", (-8, 8), "right"),
        "Fort (v9)": ("Fort (v9)", (6, -12), "left"), "mélange seul (délai 0)": ("mélange seul", (6, -12), "left"),
        "Très fort (v9)": ("Très fort (v9)", (6, -4), "left"), "délais avant (712)": ("délais avant", (-6, -14), "right"),
        "générateur désactivé": ("générateur désactivé", (6, 4), "left"),
        "délais 100 + mélange ARK": ("délais 100 + mélange ARK", (-8, 6), "right")}
for p in TOUS:
    if p[0] in ETIQ:
        lab, off, ha = ETIQ[p[0]]
        ax.annotate(lab, (surc(p[2]), p[3]), xytext=off, textcoords="offset points", fontsize=7, ha=ha,
                    color="#1f5fa8" if p[0] in noms_h else "0.45")
ax.axhline(1e4, ls=":", c="0.4"); ax.text(1, 1.15e4, "cible N_vie = 10⁴", fontsize=7, color="0.35")
ax.set_yscale("log"); ax.set_ylim(10, 1.5e5); ax.set_xlim(-2, 62)
ax.set_xlabel("surcoût par rapport au générateur désactivé (%)")
ax.set_ylabel("N : traces pour retrouver la clé dans 10 % des cas\n(meilleur des attaquants A1 à A4)")
ax.set_title("Front de Pareto validé : bleu = non dominé, gris = dominé, creux = clé non cassée (borne)",
             loc="left", fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(T10, "front_valide.pdf")); plt.show()

In [ ]:
# ==== Cellule 9.1 : flashage du firmware v10.1 (carte branchée) ====
import monitor_collect as mc
import politique_v10 as pol
for m_ in (bv, sc, no, pv, pol):
    importlib.reload(m_)
try:
    nt.scope.dis(); nt.target.dis()                  # libère l'ancienne connexion si elle existe
except Exception:
    pass
nt = bv.connecter(programmer=True)                   # programme simpleserial-aes-CWLITEARM.hex (v10.1)
INFO = bv.info_firmware(nt)
assert INFO["version"] == 11, "la carte n'exécute pas le firmware v10.1"
BUILD_V101 = bv.empreinte_firmware()
print("build v10.1 :", BUILD_V101)

### 9.2 Non-régression : chiffré AES et durées des niveaux (≈ 2 min)
Écart attendu < 1 % par rapport aux durées v10 : les résultats A4 et 50 000 traces restent alors valables pour ce build.

In [ ]:
# ==== Cellule 9.2 : non-régression — AES correct et durées des niveaux identiques à la v10 (≈ 2 min) ====
NIV, REF0 = pol.charger_niveaux()
PT0 = np.frombuffer(bytes.fromhex("6bc1bee22e409f96e93d7e117393172a"), np.uint8)
CT0 = "3ad77bb40d7a3660a89ecaf32466ef97"
cfg_niv = lambda v: no.Config(pol.COMPS, ((1, v["d"]), (3, 0)), pol.MASK)
TESTS = [("générateur désactivé (0x000)", no.Config((), (), 0x000), REF0)] + \
        [("niveau %d (d = %d)" % (v["niveau"], v["d"]), cfg_niv(v), v["cycles"]) for v in NIV]
NONREG = []
for nom, cfg, ref in TESTS:
    mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
    bv.setup_v10(cfg)(nt)
    ok = all(bytes(sc.capture_one(nt, PT0)[1]).hex() == CT0 for _ in range(10))
    t = bv.mesurer_durees(nt, cfg, n=200)
    ecart = 100 * (t.mean() / ref - 1)
    NONREG.append(dict(test=nom, aes_ok=ok, cycles=float(t.mean()), ref=ref, ecart_pct=ecart))
    print("%-30s AES %s | %7.0f cycles (réf. v10 %7.0f) | écart %+5.2f %% %s"
          % (nom, "ok  " if ok else "FAUX", t.mean(), ref, ecart, "" if abs(ecart) < 1 else "<-- À VÉRIFIER"))
json.dump(dict(build=BUILD_V101, tests=NONREG), open(os.path.join(bv.TRACE_ROOT, "nonregression_v101.json"), "w"), indent=1)

### 9.3 Politique embarquée : plancher calibré et conformité (≈ 1 min 30)

In [ ]:
# ==== Cellule 9.3 : politique embarquée — plancher calibré, budget et conformité (≈ 2 min 30) ====
CFG_POL = pol.export_header(NIV, path=None)          # mêmes paramètres que le noise_policy.h compilé
mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
mc._cmd(nt.target, "R", bytearray([0]))              # banc : oublie le budget mémorisé
bv._lire_ligne(nt, "H", [1], "H", 8)                 # banc : efface les compteurs de fautes et de redémarrages
mc._cmd(nt.target, "a", bytearray([1]))              # politique embarquée active
rng = np.random.default_rng(7)
PL, TRG = [], []
for i in range(3000):
    _, _, trig, plog = sc.capture_one(nt, rng.integers(0, 256, 16, dtype=np.uint8), with_trig=True, with_plog=True)
    PL.append(plog); TRG.append(trig)
assert all(p is not None for p in PL), "pas de journal 'P' : la politique n'est pas active"
D = [pol.lire_plog(p) for p in PL]
LV, TRG = np.array([d["level"] for d in D]), np.array(TRG, float)
print("plancher attendu : niveau %d avec p = %.4f, sinon niveau %d" % (CFG_POL["a"] + 1, CFG_POL["p"], CFG_POL["b"] + 1))
for k in range(3):
    sel = LV == k
    if sel.any():
        print("  niveau %d : %4d chiffrements (%5.1f %%) | durée %7.0f cycles (forcé : %7.0f)"
              % (k + 1, sel.sum(), 100 * sel.mean(), TRG[sel].mean(), NONREG[k + 1]["cycles"]))
print("x6 maximal : %d | classes : %s | budget après %d chiffrements : %d ‰ (attendu ≈ %.0f ‰)"
      % (max(d["x"][6] for d in D), sorted({d["classe"] for d in D}), len(D), D[-1]["x"][1], 1000 * len(D) / 1e4))
CONF = pol.conformite(PL, CFG_POL)
print("conformité :", CONF)

### 9.4 Budget conservé après reset, x6, nouvelle clé (≈ 10 s)

In [ ]:
# ==== Cellule 9.4 : budget conservé après un reset, x6 sur redémarrages répétés, nouvelle clé (≈ 10 s) ====
def un_chiffrement():
    _, _, _, plog = sc.capture_one(nt, rng.integers(0, 256, 16, dtype=np.uint8), with_trig=True, with_plog=True)
    return pol.lire_plog(plog)
def apres_reset(cle):
    # PAS de mc.prepare_target ici : il envoie 'R', qui remet volontairement le budget à zéro (commande de banc,
    # absente du firmware de production). On reproduit ce que voit la carte en usage réel : reset + même clé.
    nt.reset(); mc._flush(nt.target, 100)
    mc._cmd(nt.target, "k", bytearray(cle)); mc._cmd(nt.target, "a", bytearray([1]))
    return un_chiffrement()
avant = un_chiffrement()["x"][1]
d1 = apres_reset(bv.KEY)
print("1er reset + même clé : budget %d ‰ (avant : %d ‰) -> %s | x6 = %d, niveau %d, classe %d"
      % (d1["x"][1], avant, "conservé" if d1["x"][1] >= avant else "PERDU", d1["x"][6], d1["level"] + 1, d1["classe"]))
d2 = apres_reset(bv.KEY)
print("2e reset             : x6 = %d -> niveau %d, classe %d (attendu : x6 >= 1, niveau 3, classe 2 = faute)"
      % (d2["x"][6], d2["level"] + 1, d2["classe"]))
mc._cmd(nt.target, "k", bytearray(os.urandom(16)))   # autre clé : budget neuf
d3 = un_chiffrement()
print("autre clé            : budget %d ‰ (attendu : 0)" % d3["x"][1])
# nettoyage pour la suite : clé de référence, budget oublié, compteurs effacés, politique coupée
mc._cmd(nt.target, "k", bytearray(bv.KEY)); mc._cmd(nt.target, "R", bytearray([0]))
bv._lire_ligne(nt, "H", [1], "H", 8); mc._cmd(nt.target, "a", bytearray([0]))

### 9.5 Signaux de faute en fonctionnement normal (≈ 2 min)
Firmware v10.1 complet (build `4f9bdf0701a3`) : en plus de x6, la carte mesure la **durée nette** de chaque
chiffrement (durée moins les délais insérés) et vérifie un **compteur de rondes** (10 rondes exécutées ?).
Une ronde manquante est comptée comme une faute (entrée x6) et la sortie n'est pas livrée. Nouvelle ligne 'G'
émise avec 'H' quand le rapport de faute est actif. Cette cellule mesure ces signaux SANS glitch : c'est la
référence « usage normal » du classifieur de fautes.

In [ ]:
# ==== Cellule 9.5 : signaux de faute en fonctionnement NORMAL, sans glitch (≈ 2 min) ====
# Référence pour le classifieur : en usage normal, la durée NETTE (durée - délais insérés) doit être presque
# constante, et aucune erreur de ronde ni aucun désaccord de redondance ne doit apparaître.
import fautes_v10 as fv
importlib.reload(fv)
SIG_NORMAL = []
print("%-30s %-11s | %9s %7s %13s | %9s %7s | %s" % ("configuration", "redondance", "net moy.", "σ net", "net min-max",
      "total", "σ tot.", "erreurs ronde / désaccords / chiffrés faux"))
for nom, cfg, ref in TESTS:
    for mode in (0, 1):
        mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
        bv.setup_v10(cfg)(nt)
        bv._lire_ligne(nt, "H", [1], "H", 8)                     # compteurs à zéro
        fv.rapport_fautes(nt, mode=mode)                         # lignes H et G après chaque chiffrement
        ops = fv.signaux_normaux(nt, n=300)
        assert all(o["g"] is not None for o in ops), "pas de ligne G : firmware v10.1 (signaux de faute) non flashé ?"
        net = np.array([o["g"]["net_cycles"] for o in ops], float)
        tot = np.array([o["g"]["total_cycles"] for o in ops], float)
        exp = sc.expected_ct(bv.KEY, np.stack([o["pt"] for o in ops]))
        faux = sum(bytes(o["ct"]) != bytes(e) for o, e in zip(ops, exp))
        rer = sum(o["g"]["round_err_op"] for o in ops); mis = sum(o["g"]["mismatch"] for o in ops)
        SIG_NORMAL.append(dict(config=nom, redondance=mode, net=net.tolist(), total=tot.tolist(),
                               erreurs_ronde=int(rer), desaccords=int(mis), faux=int(faux)))
        print("%-30s %-11s | %9.0f %7.1f %6.0f-%-6.0f | %9.0f %7.0f | %d / %d / %d"
              % (nom, "toujours" if mode else "non", net.mean(), net.std(), net.min(), net.max(),
                 tot.mean(), tot.std(), rer, mis, faux))
fv.rapport_fautes(nt, mode=0, actif=False)
json.dump(dict(build=BUILD_V101, mesures=SIG_NORMAL), open(os.path.join(bv.TRACE_ROOT, "signaux_normaux_v101.json"), "w"))

## 10. Étape 3.3 — glitchs d'horloge sur l'AES (données du classifieur de fautes)
Chaque campagne alterne des blocs **sans glitch** (50 chiffrements, horloge propre) et des blocs **glitchés**
(200 essais, réglages tirés au hasard : largeur, décalage, instant). Politique embarquée active. Chaque essai est
enregistré avec ses signaux (lignes G, H, P) dans `trace/v10/fautes/<campagne>.jsonl` (reprise automatique).
Classes : normal / **détecté** (compteur de rondes ou redondance : sortie non livrée) / **exploit** (faute DFA
non détectée) / faute non détectée / reset. Aucune récupération de clé : on mesure les effets et la détection.
Glitch d'horloge uniquement (pas de MOSFET de tension à ce stade).

### 10.1 Essai court (≈ 3 min)

In [ ]:
# ==== Cellule 10.1 : essai court de glitch d'horloge (≈ 3 min) — vérifier que les glitchs ont un effet ====
import glitch_v10 as gl
importlib.reload(fv); importlib.reload(gl)
FAUTES = bv.dossier("fautes")
P_ESSAI = gl.campagne(nt, bv.KEY, "essai_court", FAUTES, mode="sans_redondance", n_glitch=400,
                      bloc_glitch=200, bloc_normal=50, seed=100)
A_ESSAI = gl.analyse(gl.charger(P_ESSAI))
for k, v in A_ESSAI.items():
    print("%-22s %s" % (k, v))

### 10.2 Campagnes : sans redondance, redondance automatique, redondance toujours (≈ 1 h)

In [ ]:
# ==== Cellule 10.2 : campagnes de glitch, 3 modes de redondance x 3000 essais glitchés (≈ 1 h au total) ====
# Reprise automatique : relancer la cellule après une interruption reprend là où elle s'est arrêtée.
CAMP_FIA = {}
for mode in ("sans_redondance", "redondance_auto", "redondance_toujours"):
    CAMP_FIA[mode] = gl.campagne(nt, bv.KEY, "horloge_" + mode, FAUTES, mode=mode, n_glitch=3000,
                                 bloc_glitch=200, bloc_normal=50, seed=1)
nt.scope.io.hs2 = "clkgen"                                   # horloge propre pour la suite

### 10.2b Campagnes ciblées (≈ 40 min)
La campagne large (10.2) sert de ligne de base (attaquant non calibré : ≈ 1 faute pour 3000 glitchs). Les
campagnes ciblées visent les zones productives trouvées par l'exploration (10.1b) : ce sont les données
d'entraînement du classifieur de fautes.

In [ ]:
# ==== Cellule 10.2b : campagnes CIBLÉES sur les zones productives de l'exploration (≈ 40 min, reprise auto) ====
# Exploration (10.1b) : effets seulement pour |largeur| ≈ 8-16 et décalage ≈ ±(36-45) ou proche de 0 ;
# répétition 10 -> presque toujours un reset, répétition 1 -> presque rien. Les fautes apparaissent à la
# frontière : on tire la répétition entre 1 et 5.
importlib.reload(gl)
ZONES = [(-18, -6, -48, -32), (-18, -6, 40, 48), (6, 18, -12, 12), (-18, -6, -12, 12)]
CAMP_CIBLE = {}
for mode in ("sans_redondance", "redondance_auto", "redondance_toujours"):
    CAMP_CIBLE[mode] = gl.campagne(nt, bv.KEY, "cible_" + mode, FAUTES, mode=mode, n_glitch=3000,
                                   bloc_glitch=200, bloc_normal=50, zones=ZONES, repeats=(1, 2, 3, 4, 5), seed=2)
nt.scope.io.hs2 = "clkgen"                                   # horloge propre pour la suite

### 10.3 Analyse (sans carte)

In [ ]:
# ==== Cellule 10.3 : résultats des glitchs et pouvoir de détection de chaque signal (sans carte) ====
import glob
import glitch_v10 as gl
importlib.reload(gl)
FAUTES = os.path.join(bv.TRACE_ROOT, "fautes")
RES_FIA = {}
for p in sorted(glob.glob(os.path.join(FAUTES, "horloge_*.jsonl")) + glob.glob(os.path.join(FAUTES, "cible_*.jsonl"))):
    RES_FIA[os.path.basename(p)[:-6]] = gl.analyse(gl.charger(p))
print("%-28s | %-44s | %7s | %s" % ("campagne", "essais glitchés : normal / détecté / exploit / faute / reset",
      "f. eff.", "vues par : ronde / redondance / durée nette | non vues | fausses alertes durée"))
for nom, a in RES_FIA.items():
    c = a["classes_glitch"]
    print("%-28s | %6d / %6d / %6d / %6d / %6d          | %7d | %d / %d / %d | %d | %d sur %d normaux"
          % (nom, c["normal"], c["detecte"], c["exploit"], c["faute"], c["reset"], a["fautes_effectives"],
             a.get("vues_par_ronde", 0), a.get("vues_par_redondance", 0), a.get("vues_par_duree", 0),
             a.get("non_vues", 0), a["fausses_alertes_duree"], a["n_normaux"]))
json.dump(RES_FIA, open(os.path.join(FAUTES, "resultats_glitch_v101.json"), "w"), indent=1)

In [ ]:
# ==== Cellule 10.1b : exploration des réglages de glitch (grille largeur x décalage x répétition, ≈ 5 min) ====
importlib.reload(gl)
P_EXPL = gl.exploration(nt, bv.KEY, FAUTES, nom="exploration_horloge")
TOP, TOT = gl.resume_exploration(P_EXPL)
print("total : %s" % TOT)
print("%8s %8s %7s | %4s %6s %6s %8s %8s %7s" % ("largeur", "décalage", "répét.", "n", "reset", "faute", "exploit", "détecté", "durée"))
for (w, o, rp), a in TOP:
    print("%8.1f %8.1f %7d | %4d %6d %6d %8d %8d %7d" % (w, o, rp, a["n"], a["reset"], a["faute"], a["exploit"], a["detecte"], a["duree"]))

## 11. Étape 4.2 — sessions d'usage normal réalistes (données « normales » du classifieur)
Quatre scénarios sans glitch, 600 chiffrements chacun, pauses irrégulières, politique et redondance automatique
actives comme en production :
- **usage** : usage normal seul ;
- **reset_isole** : un reset (broche nRST) tous les 100 chiffrements ;
- **alimentation** : une coupure d'alimentation tous les 150 chiffrements (démarrage à froid) ;
- **double_reset** : deux resets rapprochés tous les 200 chiffrements (ex. redémarrage après mise à jour).
Question : la règle actuelle (x6, 1er reset toléré) déclenche-t-elle de fausses alertes en usage normal ?
Fichiers : `trace/v10/fautes/normal_<scénario>.jsonl`.

### 11.1 Enregistrement des sessions (carte branchée, ≈ 12 min)

In [ ]:
# ==== Cellule 11.1 : sessions d'usage NORMAL réalistes, avec resets et coupures légitimes (≈ 12 min) ====
# Aucun glitch (horloge propre). Après chaque reset : clé rechargée, rapport de faute et politique réactivés,
# comme le ferait l'hôte en production ; jamais de 'R' (le budget doit survivre).
import sessions_v10 as sv
importlib.reload(fv); importlib.reload(sv)
FAUTES = bv.dossier("fautes")
SESS = {}
for k, (nom, kw) in enumerate(sv.SCENARIOS.items()):
    bv._lire_ligne(nt, "H", [1], "H", 8)                      # chaque scénario part de compteurs à zéro
    SESS[nom] = sv.session(nt, bv.KEY, nom, FAUTES, seed=300 + k, **kw)
    print("   ", sv.resume(SESS[nom]))
import monitor_collect as mc
mc._cmd(nt.target, "a", bytearray([0])); fv.rapport_fautes(nt, mode=0, actif=False)

### 11.2 – 11.4 Étape 4.3 — jeu de données, entraînement et export du détecteur de fautes
Le jeu de données réunit les campagnes de glitch (10.2, 10.3) et les sessions normales (11.1). Un simulateur, validé contre les compteurs renvoyés par la carte, rejoue les compteurs embarqués ; un arbre de décision appris décide des cas ambigus (resets), une règle de sûreté garde les preuves de faute.

In [ ]:
# ==== Cellule 11.2 : jeu de données du détecteur + fidélité du simulateur de compteurs (hors ligne, < 1 s) ====
import importlib, sys
sys.path.insert(0, "/home/khadim/chipwhisperer/projet")
import detecteur_v10 as dt; importlib.reload(dt)
import pandas as pd; pd.set_option("display.width", 250)
FAUTES = "/media/khadim/2A53DBCE3700DFDA/trace/v10/fautes"
CAMPS, NORM = dt.charger(FAUTES)
Jtout = dt.jeu(CAMPS, NORM, dt.LARGES + dt.CIBLEES, dt.NORMALES)
print("Jeu de données : %d essais | %d d'attaque observable | %d glitchés indécidables | %d normaux"
      % (len(Jtout["y"]), Jtout["lab"].sum(), (Jtout["y"] & ~Jtout["lab"]).sum(), (~Jtout["y"]).sum()))
print("Caractéristiques :"); [print("   %-5s %s" % (k, dt.DESCR[k])) for k in dt.FEATS]
print("Fidélité du simulateur (boots_recent mesuré = simulé, %) :", dt.fidelite(NORM))
# Diagnostic de la règle x6 du firmware v10.1 en usage normal
for n in dt.NORMALES:
    J = dt.jeu(CAMPS, NORM, [], [n], decroissance_persistante=False)
    print("   %-13s fausses alertes règle x6 v10.1 : %5.1f %% des opérations" % (n, 100 * dt.regle_x6(J["X"]).mean()))


In [ ]:
# ==== Cellule 11.3 : validation croisée sur données jamais vues — règle x6 vs modèle appris (hors ligne, < 2 s) ====
# A : apprentissage glitchs LARGES + usage + reset_isole  ->  test glitchs CIBLÉS + double_reset + alimentation
# B : apprentissage glitchs CIBLÉS + double_reset + alimentation  ->  test glitchs LARGES + usage + reset_isole
# Hyperparamètres fixés a priori : profondeur 3, coûts égaux (kFA 1), pas de maintien, arbre MONOTONE
# (plus de symptômes => jamais moins d'alerte). Sensibilité complète en annexe (sensibilite_detecteur.csv).
import numpy as np, matplotlib.pyplot as plt
V = dt.validation_croisee(CAMPS, NORM, profondeur=3, k_fa=1, hold=0)
print(V.to_string(index=False))
S = dt.sensibilite(CAMPS, NORM); S.to_csv("/media/khadim/2A53DBCE3700DFDA/trace/v10/fautes/sensibilite_detecteur.csv", index=False)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
lab = {"règle x6 (firmware v10.1)": "règle x6 v10.1", "règle x6, décroissance corrigée": "x6 corrigée"}
for ax, s in zip(axs, "AB"):
    d = V[V.split == s]; fa_cols = [c for c in d.columns if c.startswith("FA ") and "campagne" not in c and d[c].notna().all()]
    noms = [lab.get(x, "sûreté + arbre appris") for x in d["détecteur"]]
    x = np.arange(len(d)); w = 0.8 / (len(fa_cols) + 1)
    for k, c in enumerate(fa_cols):
        ax.bar(x + k * w, d[c], w, label=c.replace(" %", "").replace("FA ", "fausses alertes : "))
    ax.bar(x + len(fa_cols) * w, d["délai médian (essais)"], w, color="0.4", label="délai médian de détection (essais)")
    ax.set_xticks(x + w * len(fa_cols) / 2); ax.set_xticklabels(noms, fontsize=8)
    ax.set_title("Split %s (test : %s)" % (s, "glitchs ciblés" if s == "A" else "glitchs larges"), fontsize=10)
    ax.set_ylabel("% des opérations normales / essais"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig("/media/khadim/2A53DBCE3700DFDA/trace/v10/fautes/detecteur_validation.png", dpi=200); plt.show()


In [ ]:
# ==== Cellule 11.4 : ARBRE UNIQUE de politique (comme en v9), entraînement, validation, export C (hors ligne) ====
# Un seul arbre, exporté dans noise_policy_v12.h (dans projet/, PAS encore dans le firmware) :
#   1. règle de sûreté (fixée)   : faute prouvée (désaccord, erreur de rondes, durée)  -> niveau 3 + redondance
#   2. sous-arbre APPRIS         : resets suspects (rs16, rs64, boots, gap)          -> niveau 3 + redondance
#   3. nœud budget DISTILLÉ      : budget épuisé (appris sur les décisions du planificateur, comme en v9) -> niveau 3
#   4. sinon                     : plancher calibré (mélange niveaux 2/3)
import politique_v10 as pv; importlib.reload(pv); importlib.reload(dt)
from sklearn.tree import export_text
# (a) validation croisée de l'arbre unique complet, sur données jamais vues
VU = dt.validation_arbre_unique(CAMPS, NORM)
print(VU.to_string(index=False))
# (b) modèle final : toutes les données
AU = dt.construire_arbre_unique(Jtout)
print("\nNœud budget distillé : seuil appris x1 <= %d ‰ -> non épuisé (accord avec le planificateur %.2f %%)"
      % (AU["budget"]["seuil"], 100 * AU["budget"]["accord"]))
print("Sous-arbre appris sur les resets :"); print(export_text(AU["resets"], feature_names=dt.FEATS))
# (c) export C (projet/noise_policy_v12.h) et contrôle C = Python
NIV, REF0 = pv.charger_niveaux()
EXP = dt.export_politique(AU, NIV, n_vie=10000, m=2,
                          commentaire="Entraîné sur 6 campagnes de glitch + 4 sessions normales (cellules 10.2, 10.2b, 11.1).")
s = EXP["header"]; i = s.index("static uint8_t policy_decide("); print(s[i:s.index("#endif", i)])
ok, n = dt.verifier_c(AU, EXP, Jtout)
print("Contrôle C = Python : %d/%d états identiques -> %s" % (ok, n, "CONFORME" if ok == n else "NON CONFORME"))
print("Fichier écrit :", EXP["chemin"], "(le firmware n'est pas modifié)")


## 12. Firmware v12 — modèle appris déployé : entraînement, validation sur la carte (FIA et SCA)
Ordre : 12.0 (hors ligne) → 12.1 à 12.6 (carte branchée, ≈ 45 min) → 12.7 capture SCA (≈ 40 min) →
12.8 attaques en arrière-plan (sans carte) → 12.9 résultats → 12.10 simulation escalier / mélange (hors ligne).

In [ ]:
# ==== Cellule 12.0 : ENTRAÎNEMENT du modèle déployé, pas à pas (hors ligne, < 5 s) ====
import detecteur_v10 as dt, politique_v10 as pv
importlib.reload(dt); importlib.reload(pv)
import matplotlib.pyplot as plt
FAUTES = os.path.join(bv.TRACE_ROOT, "fautes")
FW_DIR = "/home/khadim/chipwhisperer/firmware/mcu6/simpleserial-aes"
CAMPS, NORM = dt.charger(FAUTES)                                   # 6 campagnes de glitch + 4 sessions normales
Jtout = dt.jeu(CAMPS, NORM, dt.LARGES + dt.CIBLEES, dt.NORMALES)   # compteurs embarqués rejoués sur chaque essai
fig, ax = plt.subplots(figsize=(11, 5))
AU = dt.entrainement_detaille(Jtout, profondeur=3, k_fa=1, ax=ax)  # <-- l'apprentissage (scikit-learn .fit)
ax.set_title("Sous-arbre appris sur les resets (déployé dans le firmware v12)", fontsize=10)
fig.savefig(os.path.join(FAUTES, "arbre_appris.pdf")); plt.show()
NIV, REF0 = pv.charger_niveaux()
EXP = dt.export_politique(AU, NIV, n_vie=10000, m=2,
                          commentaire="Entraîné sur 6 campagnes de glitch + 4 sessions normales (cellules 10.2, 10.2b, 11.1).")
ok, n = dt.verifier_c(AU, EXP, Jtout)
print("4) EXPORT C : %s | contrôle C = Python : %d/%d états identiques" % (EXP["chemin"], ok, n))
print("   identique au noise_policy.h compilé dans le firmware v12 :",
      open(EXP["chemin"]).read() == open(os.path.join(FW_DIR, "noise_policy.h")).read())

In [ ]:
# ==== Cellule 12.1 : flashage du firmware v12 (carte branchée, ≈ 1 min) ====
import monitor_collect as mc, fautes_v10 as fv, glitch_v10 as gl, sessions_v10 as sv, politique_v10 as pol
for m_ in (bv, sc, no, pol, fv, gl, sv):
    importlib.reload(m_)
try:
    nt.scope.dis(); nt.target.dis()
except Exception:
    pass
nt = bv.connecter(programmer=True)                   # programme simpleserial-aes-CWLITEARM.hex (v12)
INFO = bv.info_firmware(nt)
assert INFO["version"] == 12, "la carte n'exécute pas le firmware v12"
BUILD_V12 = bv.empreinte_firmware()
print("build v12 :", BUILD_V12)

In [ ]:
# ==== Cellule 12.2 : non-régression v12 — AES correct et durées des niveaux identiques (≈ 2 min) ====
# Si les durées sont à moins de 1 % des références, les résultats des attaques CPA (sections 8.4-8.8) restent valables.
PT0 = np.frombuffer(bytes.fromhex("6bc1bee22e409f96e93d7e117393172a"), np.uint8)
CT0 = "3ad77bb40d7a3660a89ecaf32466ef97"
cfg_niv = lambda v: no.Config(pol.COMPS, ((1, v["d"]), (3, 0)), pol.MASK)
TESTS = [("générateur désactivé (0x000)", no.Config((), (), 0x000), REF0)] + \
        [("niveau %d (d = %d)" % (v["niveau"], v["d"]), cfg_niv(v), v["cycles"]) for v in NIV]
NONREG12 = []
for nom, cfg, ref in TESTS:
    mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
    bv.setup_v10(cfg)(nt)
    ok_aes = all(bytes(sc.capture_one(nt, PT0)[1]).hex() == CT0 for _ in range(10))
    t = bv.mesurer_durees(nt, cfg, n=200)
    ecart = 100 * (t.mean() / ref - 1)
    NONREG12.append(dict(test=nom, aes_ok=ok_aes, cycles=float(t.mean()), ref=ref, ecart_pct=ecart))
    print("%-30s AES %s | %7.0f cycles (réf. %7.0f) | écart %+5.2f %% %s"
          % (nom, "ok  " if ok_aes else "FAUX", t.mean(), ref, ecart, "" if abs(ecart) < 1 else "<-- À VÉRIFIER"))
json.dump(dict(build=BUILD_V12, tests=NONREG12), open(os.path.join(bv.TRACE_ROOT, "nonregression_v12.json"), "w"), indent=1)

In [ ]:
# ==== Cellule 12.3 : arbre unique sur la carte — plancher, budget et CONFORMITÉ v12 (≈ 2 min 30) ====
# La carte écrit dans son journal 'P' les compteurs qu'elle a lus pour décider ; Python rejoue l'arbre sur ces
# compteurs : 100 % des décisions déterministes doivent coïncider, et la part du niveau 2 doit valoir p.
CFG12 = dict(a=EXP["cfg"]["a"], b=EXP["cfg"]["b"], p16=EXP["cfg"]["p16"])
mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
mc._cmd(nt.target, "R", bytearray([0]))              # banc : oublie le budget mémorisé
bv._lire_ligne(nt, "H", [1], "H", 8)                 # banc : compteurs de faute à zéro
mc._cmd(nt.target, "a", bytearray([1]))              # politique embarquée (arbre unique)
rng = np.random.default_rng(7)
PL12, TRG12 = [], []
for i in range(3000):
    _, _, trig, plog = sc.capture_one(nt, rng.integers(0, 256, 16, dtype=np.uint8), with_trig=True, with_plog=True)
    PL12.append(plog); TRG12.append(trig)
assert all(p is not None for p in PL12), "pas de journal 'P' : la politique n'est pas active"
D12 = [dt.lire_plog_v12(p) for p in PL12]
LV12, TRG12 = np.array([d["level"] for d in D12]), np.array(TRG12, float)
for k in range(3):
    sel = LV12 == k
    if sel.any():
        print("  niveau %d : %4d chiffrements (%5.1f %%) | durée %7.0f cycles (forcé : %7.0f)"
              % (k + 1, sel.sum(), 100 * sel.mean(), TRG12[sel].mean(), NONREG12[k + 1]["cycles"]))
print("classes : %s | alertes faute : %d (attendu 0 : aucune faute, aucun reset) | budget après %d chiffrements : %d ‰"
      % (sorted({d["classe"] for d in D12}), sum(d["alerte"] for d in D12), len(D12), D12[-1]["x1"]))
CONF12 = dt.conformite_v12(PL12, AU, CFG12)
print("conformité v12 :", {k: v for k, v in CONF12.items() if k != "exemples"})

In [ ]:
# ==== Cellule 12.4 : resets — budget conservé, alerte courte après un reset, alerte durable sur resets répétés (≈ 20 s) ====
def un_chiffrement():
    _, _, _, plog = sc.capture_one(nt, rng.integers(0, 256, 16, dtype=np.uint8), with_trig=True, with_plog=True)
    return dt.lire_plog_v12(plog)
def apres_reset(cle):
    nt.reset(); mc._flush(nt.target, 100)                    # PAS de prepare_target (il envoie 'R')
    mc._cmd(nt.target, "k", bytearray(cle)); mc._cmd(nt.target, "a", bytearray([1]))
    return un_chiffrement()
fmt = lambda d: "classe %d, niveau %d, gap %d, boots %d, budget %d ‰" % (d["classe"], d["level"] + 1, d["feats"][3], d["feats"][2], d["x1"])
avant = un_chiffrement()
print("avant reset            :", fmt(avant))
d1 = apres_reset(bv.KEY)
print("juste après 1 reset     :", fmt(d1), "| attendu : classe 2 (reset récent), budget conservé")
suite = [un_chiffrement() for _ in range(14)]
print("14 chiffrements après   :", fmt(suite[-1]), "| attendu : classe 0 (retour au plancher)")
for _ in range(4):
    d4 = apres_reset(bv.KEY)
suite4 = [un_chiffrement() for _ in range(20)]
print("après 4 resets rapprochés + 20 chiffrements :", fmt(suite4[-1]), "| attendu : classe 2 (boots = 4)")
mc._cmd(nt.target, "k", bytearray(os.urandom(16)))
print("autre clé               : budget %d ‰ (attendu 0)" % un_chiffrement()["x1"])
mc._cmd(nt.target, "k", bytearray(bv.KEY)); mc._cmd(nt.target, "R", bytearray([0]))
bv._lire_ligne(nt, "H", [1], "H", 8); mc._cmd(nt.target, "a", bytearray([0]))

In [ ]:
# ==== Cellule 12.5 : sessions d'usage NORMAL rejouées avec le firmware v12 (≈ 12 min) ====
# Mêmes 4 scénarios que 11.1, nouveaux fichiers normal_v12_<scénario>.jsonl (ceux de 11.1 sont conservés).
# Prédiction (arbre rejoué sur les sessions 11.1) comparée à la mesure sur la carte, + fidélité des compteurs.
SESS12 = {}
for k, (nom, kw) in enumerate(sv.SCENARIOS.items()):
    bv._lire_ligne(nt, "H", [1], "H", 8)
    SESS12[nom] = sv.session(nt, bv.KEY, "v12_" + nom, FAUTES, seed=300 + k, **kw)
mc._cmd(nt.target, "a", bytearray([0])); fv.rapport_fautes(nt, mode=0, actif=False)
print("%-13s | %-28s | %-28s | %s" % ("scénario", "alerte faute MESURÉE (carte)", "alerte PRÉDITE (simulateur)", "fidélité compteurs"))
for nom in sv.SCENARIOS:
    R = [json.loads(l) for l in open(SESS12[nom])]
    cl = [dt.lire_plog_v12(r["plog"])["classe"] for r in R if r.get("g") and r.get("plog")]
    Jn = dt.jeu(CAMPS, NORM, [], [nom])
    pred = dt.preuve_faute(Jn["X"]) | AU["resets"].predict(Jn["X"]).astype(bool)
    print("%-13s | %26.1f %% | %26.1f %% | %s" % (nom, 100 * np.mean(np.array(cl) == 2), 100 * pred.mean(),
                                                    dt.fidelite_v12(SESS12[nom])))

In [ ]:
# ==== Cellule 12.6 : courte campagne de glitch CIBLÉE avec le firmware v12 (≈ 15 min, reprise auto) ====
ZONES = [(-18, -6, -48, -32), (-18, -6, 40, 48), (6, 18, -12, 12), (-18, -6, -12, 12)]
bv._lire_ligne(nt, "H", [1], "H", 8)
P12 = gl.campagne(nt, bv.KEY, "v12_cible_auto", FAUTES, mode="redondance_auto", n_glitch=1000,
                  bloc_glitch=200, bloc_normal=50, zones=ZONES, repeats=(1, 2, 3, 4, 5), seed=12)
nt.scope.io.hs2 = "clkgen"
R12 = gl.charger(P12)
A12 = gl.analyse(R12)
print("classes des essais glitchés :", A12["classes_glitch"], "| fautes effectives :", A12["fautes_effectives"])
# délai de détection par bloc et fausses alertes, lus dans les décisions de la CARTE (journal 'P')
blocs, b, prev = [], -1, None
for r in R12:
    if "glitch" not in r: continue
    if r["glitch"] != prev: b += 1; prev = r["glitch"]; blocs.append(dict(glitch=r["glitch"], cl=[]))
    blocs[-1]["cl"].append(dt.lire_plog_v12(r["plog"])["classe"] if r.get("plog") else None)
delais = [next((i + 1 for i, c in enumerate(B["cl"]) if c == 2), None) for B in blocs if B["glitch"]]
fa = [np.mean([c == 2 for c in B["cl"] if c is not None]) for B in blocs if not B["glitch"]]
print("blocs glitchés détectés : %d/%d | délai médian %s essais | blocs normaux : alerte %.1f %% du temps (vigilance après attaque)"
      % (sum(d is not None for d in delais), len(delais), np.median([d for d in delais if d]), 100 * np.mean(fa)))
print("fidélité des compteurs (carte vs simulateur) :", dt.fidelite_v12(P12))

In [ ]:
# ==== Cellule 12.7 : SCA de bout en bout — 20 000 traces avec la POLITIQUE ACTIVE (≈ 40 min, reprise auto) ====
# Comme en usage réel : la carte choisit elle-même son niveau (plancher calibré, puis niveau 3 quand le budget de
# 10 000 chiffrements est épuisé). Aucune configuration forcée. Les journaux 'P' de chaque trace sont enregistrés.
import glob
NOM_BEB = "politique_v12_20k"
VALD = bv.dossier("validation")
build = bv.empreinte_firmware()
bv.archiver_si_autre_build(os.path.join(VALD, NOM_BEB), build); bv._marquer_build(os.path.join(VALD, NOM_BEB), build)
def setup_politique(nt_):
    sc.make_setup()(nt_)                                   # remise à zéro : aucun forçage
    bv.regler_fm(nt_, bv.FM_DEFAUT)
    bv._lire_ligne(nt_, "H", [1], "H", 8)                  # compteurs de faute à zéro
    mc._cmd(nt_.target, "a", bytearray([1]))               # la politique embarquée décide seule
META = dict(firmware=12, build=build, fm_shift=int(bv.FM_DEFAUT), politique=True,
            config=dict(comps=[], raw=[], mask=0, name="politique_v12 (arbre unique)"))
P_BEB = sc.capture_campaign(nt, NOM_BEB, 20000, bv.KEY, seed=3, outdir=VALD, chunk=1000, setup=setup_politique, meta=META)
mc._cmd(nt.target, "a", bytearray([0]))
PLB = np.load(os.path.join(P_BEB, "plog.npy"))           # journaux 'P' regroupés à la fin de la capture
DB = [dt.lire_plog_v12(p) for p in PLB]
lvb = np.array([d["level"] for d in DB]); clb = np.array([d["classe"] for d in DB])
for h, sl in (("chiffrements 1-10 000", slice(0, 10000)), ("chiffrements 10 001-20 000", slice(10000, 20000))):
    print("%-27s : niveau 2 %5.1f %% | niveau 3 %5.1f %% | classes %s"
          % (h, 100 * np.mean(lvb[sl] == 1), 100 * np.mean(lvb[sl] == 2), dict(zip(*np.unique(clb[sl], return_counts=True)))))
print("budget épuisé au chiffrement n°", int(np.argmax(clb == 1)) + 1 if (clb == 1).any() else "jamais")

In [ ]:
# ==== Cellule 12.8 : attaques A1-A4 sur la campagne de bout en bout, en arrière-plan (sans carte, ≈ 1-3 h) ====
import subprocess
CMD = "%s attaque_bout_en_bout_v12.py > %s/log_bout_en_bout_v12.txt 2>&1" % (sys.executable, VALD)
proc_beb = subprocess.Popen(["sh", "-c", CMD], cwd=bv.PROJET, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
                            start_new_session=True)
print("attaques lancées, PID", proc_beb.pid, "| suivi :", os.path.join(VALD, "log_bout_en_bout_v12.txt"))

In [ ]:
print(open(os.path.join(VALD, "log_bout_en_bout_v12.txt")).read()[-500:])

In [ ]:
# ==== Cellule 12.9 : résultats de l'attaque de bout en bout (quand le journal affiche FINI) ====
print(open(os.path.join(VALD, "log_bout_en_bout_v12.txt")).read()[-1500:])
RB = json.load(open(os.path.join(VALD, NOM_BEB + "_resultat_w300.json")))
print("Politique v12 déployée : X médian = %s | effort médian à %d traces = %.1f bits | N(10 %%) = %s"
      % (RB["X_median"] or "> %d" % RB["ns"][-1], RB["ns"][-1], RB["effort_med"][-1], pol.n10(RB) or "> %d" % RB["ns"][-1]))
for v in NIV:
    print("   rappel niveau %d forcé : N(10 %%) %s%d" % (v["niveau"], ">= " if v["borne"] else "", v["N"]))

In [ ]:
# ==== Cellule 12.10 : étape 4.1 — escalier de niveaux contre mélange constant (simulation, hors ligne) ====
kk, ESC = dt.simulation_escalier(NIV, n_vie=10000, m=2, p16=EXP["cfg"]["p16"], a=EXP["cfg"]["a"], b=EXP["cfg"]["b"])
fig, ax = plt.subplots(figsize=(7, 4))
for nom, o in ESC.items():
    ax.plot(kk, o["L"], label="%s : coût %.1f %%, marge réelle %.2f" % (nom, o["cout"], o["marge"]))
    print("%-28s coût moyen %.1f %% | fuite cumulée après 1 000 : %.3f | finale : %.3f | marge réelle %.2f"
          % (nom, o["cout"], o["L_1000"], o["L_final"], o["marge"]))
ax.axhline(0.5, color="0.4", ls="--", lw=0.8); ax.text(200, 0.52, "limite 1/m (m = 2)", fontsize=8)
ax.set_xlabel("chiffrements depuis le chargement de la clé"); ax.set_ylabel("fuite cumulée  Σ 1/N")
ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "escalier_melange.pdf")); plt.show()

## 13. Comparaison finale : notre modèle contre les contre-mesures classiques, à coût égal
**Notre modèle** = générateur (niveaux conçus par ParEGO) + arbre unique appris qui le pilote (firmware v12).
Le **niveau 3 statique** sépare l'apport de la conception (ParEGO) de celui du pilotage (IA).
SCA : chaque contre-mesure classique est réglée au coût du niveau 3, mêmes attaquants A1-A4, mêmes textes clairs.
FIA : redondance toujours active (classique) contre redondance déclenchée par l'arbre.
Ordre : 13.1 (carte, ≈ 10 min) → 13.2 (carte, ≈ 70 min) → 13.3 (attaques la nuit) → 13.4 → 13.5 → 13.6.

In [ ]:
# ==== Cellule 13.1 : calibration « à coût égal » des contre-mesures classiques + coût de la redondance (≈ 10 min) ====
import fautes_v10 as fv; importlib.reload(fv)
MASQUE = 0x403                                      # rondes 0, 1, 10 : comme nos niveaux (comparaison équitable)
CLASSIQUES = {                                      # (fabrique intensité -> config, bornes de la dichotomie)
    "délais aléatoires":               (lambda d: no.Config(((1, 1), (4, 1)), ((1, int(d)),), MASQUE), (1, 2000)),
    "opérations factices (type dual-rail)": (lambda r: no.Config(((2, 2),), ((2, int(r)),), MASQUE), (1, 8000)),
    "bruit ALU":                       (lambda r: no.Config(((3, 4),), ((4, int(r)),), MASQUE), (1, 8000)),
}
fv.rapport_fautes(nt, mode=0, actif=False)          # pas de redondance pendant la calibration
base12 = bv.mesurer_durees(nt, no.Config((), (), 0x000), n=200).mean()
niv3 = [v for v in NIV if v["niveau"] == 3][0]
cfg3 = no.Config(pol.COMPS, ((1, niv3["d"]), (3, 0)), pol.MASK)
c3 = bv.mesurer_durees(nt, cfg3, n=200).mean()
CIBLE = 100 * (c3 / base12 - 1)
print("coût de référence : générateur coupé %.0f cycles | niveau 3 %.0f cycles -> cible %+.1f %%" % (base12, c3, CIBLE))
CALIB = {}
for nom, (fab, bornes) in CLASSIQUES.items():
    print("==", nom)
    r, pct = bv.calibrer(nt, fab, CIBLE, base12, bornes=bornes, n=150)
    CALIB[nom] = dict(intensite=int(r), surcout_pct=float(pct), config=fab(r).to_dict())
# coût de la redondance classique (toujours active) : durée du trigger, qui couvre les DEUX exécutions de l'AES
fv.rapport_fautes(nt, mode=1, actif=False)
c3_red = bv.mesurer_durees(nt, cfg3, n=200).mean()
fv.rapport_fautes(nt, mode=0, actif=False)
COUTS = dict(base=base12, niveau3=c3, niveau3_redondance=c3_red, surcout_redondance_pct=100 * (c3_red / c3 - 1))
print("redondance toujours active : niveau 3 %.0f -> %.0f cycles (%+.1f %% de plus)" % (c3, c3_red, COUTS["surcout_redondance_pct"]))
json.dump(dict(cible_pct=CIBLE, classiques=CALIB, couts=COUTS, build=bv.empreinte_firmware()),
          open(os.path.join(bv.TRACE_ROOT, "classiques_v12.json"), "w"), indent=1)

In [ ]:
# ==== Cellule 13.2 : captures des 3 contre-mesures classiques, 20 000 traces chacune (≈ 70 min, reprise auto) ====
# Graine 0 : mêmes textes clairs que la campagne du niveau 3 (validation, série s0) -> comparaison appariée.
CLS = json.load(open(os.path.join(bv.TRACE_ROOT, "classiques_v12.json")))
DOSS_CL = bv.dossier("validation", "classiques")
NOMS_CL = {"délais aléatoires": "delais_aleatoires_20k", "opérations factices (type dual-rail)": "operations_factices_20k",
           "bruit ALU": "bruit_alu_20k"}
INDEX_CL = {}
for nom, d in CLS["classiques"].items():
    cfg = no.Config.from_dict(d["config"])
    print("==", nom, "|", cfg.name, "| surcoût calibré %+.1f %%" % d["surcout_pct"])
    INDEX_CL[nom] = bv.capturer(nt, NOMS_CL[nom], 20000, cfg=cfg, sous_dossier=os.path.join("validation", "classiques"), seed=0)
json.dump(INDEX_CL, open(os.path.join(DOSS_CL, "index.json"), "w"), indent=1)

In [ ]:
# ==== Cellule 13.3 : attaques A1-A4 des 3 campagnes classiques, en arrière-plan (sans carte, ≈ 1 h 30) ====
import subprocess
CMD = "%s attaque_classiques_v12.py > %s/log_classiques_v12.txt 2>&1" % (sys.executable, DOSS_CL)
proc_cl = subprocess.Popen(["sh", "-c", CMD], cwd=bv.PROJET, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
                           start_new_session=True)
print("attaques lancées, PID", proc_cl.pid, "| suivi :", os.path.join(DOSS_CL, "log_classiques_v12.txt"))
print("NE PAS relancer cette cellule tant que le journal n'affiche pas FINI.")

In [ ]:
print(open(os.path.join(DOSS_CL, "log_classiques_v12.txt")).read()[-500:])

In [ ]:
# ==== Cellule 13.4 : tableau SCA final à coût égal + figure coût / sécurité (quand le journal affiche FINI) ====
import matplotlib.pyplot as plt
import detecteur_v10 as dt, politique_v10 as pol
NIV, REF0 = pol.charger_niveaux()
DOSS_CL = os.path.join(bv.TRACE_ROOT, "validation", "classiques")
INDEX_CL = json.load(open(os.path.join(DOSS_CL, "index.json")))      # écrit par 13.2
print(open(os.path.join(DOSS_CL, "log_classiques_v12.txt")).read()[-600:])
V = os.path.join(bv.TRACE_ROOT, "validation"); C_ = os.path.join(bv.TRACE_ROOT, "campagnes")
CAND = "permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403"
RV = json.load(open(os.path.join(V, "resultats_validation.json")))
LIGNES = [  # (nom, type, résultat)
    ("aucune protection", "référence", json.load(open(os.path.join(C_, "sans_bruit_v10_resultat_w300.json")))),
    ("mélange seul (shuffling)", "classique", json.load(open(os.path.join(C_, "melange_20k_resultat_w300.json")))),
] + [(nom, "classique", json.load(open(p.rstrip("/") + "_resultat_w300.json"))) for nom, p in INDEX_CL.items()] + [
    ("notre niveau 3 statique (ParEGO), s0", "nôtre", RV["s0"][CAND]),
    ("notre niveau 3 statique (ParEGO), s1", "nôtre", RV["s1"][CAND]),
    ("NOTRE MODÈLE déployé (arbre v12)", "nôtre", json.load(open(os.path.join(V, "politique_v12_20k_resultat_w300.json")))),
]
TAB_SCA = []
print("%-40s %-10s | %8s | %10s | %22s | %s" % ("contre-mesure", "type", "surcoût", "N (10 %)", "effort à 8000 : méd. / p10", "octets trouvés à 8000"))
for nom, typ, r in LIGNES:
    s = 100 * (r["trig_mean"] / REF0 - 1); n10 = pol.n10(r)
    TAB_SCA.append(dict(contre_mesure=nom, type=typ, surcout_pct=s, N10=n10, effort_med_8000=r["effort_med"][-1],
                        effort_p10_8000=r["effort_p10"][-1], octets_8000=r["octets"][-1]))
    print("%-40s %-10s | %+7.1f %% | %10s | %10.1f / %5.1f bits | %.1f / 16"
          % (nom, typ, s, n10 if n10 else "> %d" % r["ns"][-1], r["effort_med"][-1], r["effort_p10"][-1], r["octets"][-1]))
json.dump(TAB_SCA, open(os.path.join(bv.TRACE_ROOT, "comparaison_sca_v12.json"), "w"), indent=1)
fig, ax = plt.subplots(figsize=(7, 4.2))
coul = {"référence": "0.5", "classique": "tab:blue", "nôtre": "tab:red"}
for t in TAB_SCA:
    ax.scatter(t["surcout_pct"], t["effort_p10_8000"], color=coul[t["type"]], s=40, zorder=3)
    ax.annotate(t["contre_mesure"], (t["surcout_pct"], t["effort_p10_8000"]), fontsize=6.5, xytext=(4, 3), textcoords="offset points")
ax.axhline(32, color="0.4", ls="--", lw=0.8); ax.text(1, 34, "clé cassée (≤ 32 bits)", fontsize=7)
ax.set_xlabel("surcoût en cycles (%)"); ax.set_ylabel("effort restant à 8 000 traces, 10 % pire cas (bits)")
ax.set_title("SCA à coût égal : classiques (bleu) contre notre modèle (rouge)", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "comparaison_sca_v12.pdf")); plt.show()

In [ ]:
# ==== Affichage des campagnes comparées en 13.4 (SCA à coût égal) — hors ligne ====
import os, json
import numpy as np
import matplotlib.pyplot as plt
import politique_v10 as pol

NIV, REF0 = pol.charger_niveaux()      # REF0 : cycles, générateur désactivé (0x000), comme en 13.4

TR  = bv.TRACE_ROOT
V   = os.path.join(TR, "validation")
B   = json.load(open(os.path.join(TR, "base_v10.json")))
INDEX_CL = json.load(open(os.path.join(V, "classiques", "index.json")))   # écrit par 13.2
V3S = ("permutation_subbytes_pre0__permutation_mixcolumns0__jitter_addroundkey_pre536"
       "__permutation_addroundkey_pre0_m403")

# (nom, type, dossier de la campagne)
CAMPS = [
    ("aucune protection",                  "référence", B["ref"]),
    ("mélange seul (shuffling)",           "classique", os.path.join(TR, "campagnes", "melange_20k")),
] + [(nom, "classique", p.rstrip("/")) for nom, p in INDEX_CL.items()] + [
    ("notre niveau 3 statique, s0",        "nôtre", os.path.join(V, V3S)),
    ("notre niveau 3 statique, s1",        "nôtre", os.path.join(V, "s1", V3S)),
    ("NOTRE MODÈLE déployé (arbre v12)",   "nôtre", os.path.join(V, "politique_v12_20k")),
]
COUL = {"référence": "0.4", "classique": "tab:blue", "nôtre": "tab:red"}

for nom, typ, p in CAMPS:                                   # contrôle des chemins avant de tracer
    if not os.path.exists(os.path.join(p, "traces.npy")):
        print("MANQUANT :", nom, "->", p)

def lire(p):
    X = np.load(os.path.join(p, "traces.npy"), mmap_mode="r")     # (N, S) int16, lu à la demande
    return X, np.load(os.path.join(p, "trig.npy"))

N_SHOW, N_STAT, ZOOM, ECH = 8, 2000, (300, 6000), 1024.0

# --- Figure 1 : traces individuelles, trace entière | zoom ---------------------------
fig, axs = plt.subplots(len(CAMPS), 2, figsize=(15, 1.9 * len(CAMPS)), sharex="col")
for i, (nom, typ, p) in enumerate(CAMPS):
    if not os.path.exists(os.path.join(p, "traces.npy")):
        continue
    X, trig = lire(p)
    for k in np.linspace(0, len(X) - 1, N_SHOW, dtype=int):
        y = np.asarray(X[k], dtype=np.float32) / ECH
        axs[i, 0].plot(y, lw=0.3, alpha=0.7)
        axs[i, 1].plot(np.arange(*ZOOM), y[ZOOM[0]:ZOOM[1]], lw=0.4, alpha=0.7)
    axs[i, 0].set_ylabel(nom, fontsize=6, color=COUL[typ])
    axs[i, 0].set_title("%d traces × %d éch. | %.0f cycles en moyenne (%+.1f %%)"
                        % (len(X), X.shape[1], trig.mean(), 100 * (trig.mean() / REF0 - 1)),
                        loc="left", fontsize=6.5)
axs[0, 1].set_title("zoom %d–%d" % ZOOM, loc="left", fontsize=7)
axs[-1, 0].set_xlabel("échantillon"); axs[-1, 1].set_xlabel("échantillon")
fig.tight_layout(); plt.show()

# --- Figure 2 : moyenne, écart-type, histogramme des durées --------------------------
fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
for nom, typ, p in CAMPS:
    if not os.path.exists(os.path.join(p, "traces.npy")):
        continue
    X, trig = lire(p)
    sub = np.asarray(X[:N_STAT, ZOOM[0]:ZOOM[1]], dtype=np.float32) / ECH
    ls = "--" if nom.endswith("s1") else "-"
    kw = dict(lw=0.6, color=COUL[typ], ls=ls, alpha=0.8)
    axs[0].plot(np.arange(*ZOOM), sub.mean(0), **kw)
    axs[1].plot(np.arange(*ZOOM), sub.std(0), label=nom, **kw)
    axs[2].hist(trig, bins=80, histtype="step", lw=1.1, color=COUL[typ], ls=ls)
axs[0].set(title="Trace moyenne (%d traces)" % N_STAT, xlabel="échantillon")
axs[1].set(title="Écart-type par échantillon", xlabel="échantillon"); axs[1].legend(fontsize=5.5)
axs[2].set(title="Durée d'un chiffrement", xlabel="cycles"); axs[2].set_yscale("log")
fig.tight_layout(); plt.show()

In [ ]:
# ==== Cellule 13.5 : tableau FIA final — redondance classique contre redondance pilotée par l'arbre (sans carte) ====
import detecteur_v10 as dt
FAU = os.path.join(bv.TRACE_ROOT, "fautes")
RG = json.load(open(os.path.join(FAU, "resultats_glitch_v101.json")))
CO = json.load(open(os.path.join(bv.TRACE_ROOT, "classiques_v12.json")))["couts"]
x_red = CO["surcout_redondance_pct"] / 100                 # surcoût relatif d'un chiffrement redondant
def cout_usage(part_alerte):                                # coût moyen en usage normal, relatif au niveau 3 sans redondance
    return 100 * part_alerte * x_red
def part_alerte_session(nom):
    R = [json.loads(l) for l in open(os.path.join(FAU, "normal_v12_%s.jsonl" % nom))]
    cl = [dt.lire_plog_v12(r["plog"])["classe"] for r in R if r.get("g") and r.get("plog")]
    return float(np.mean(np.array(cl) == 2))
ALERTE = {n: part_alerte_session(n) for n in ("usage", "reset_isole", "alimentation", "double_reset")}
# couverture de l'attaque par la redondance (campagne v12 : part des essais glitchés décidés en classe 2)
R12 = [json.loads(l) for l in open(os.path.join(FAU, "v12_cible_auto.jsonl"))]
cov12 = np.mean([dt.lire_plog_v12(r["plog"])["classe"] == 2 for r in R12 if r.get("glitch") and r.get("plog")])
def ligne(c):
    return "%d exploit(s) non détecté(s), %d faute(s) détectée(s), %d resets" % (c["exploit"] + c["faute"], c["detecte"], c["reset"])
print("Glitchs d'horloge, 3 000 essais par campagne (v10.1, mêmes réglages et même graine dans chaque mode)")
for fam in ("horloge", "cible"):
    print("  %-7s | sans redondance : %s" % (fam, ligne(RG[fam + "_sans_redondance"]["classes_glitch"])))
    print("  %-7s | toujours active : %s" % (fam, ligne(RG[fam + "_redondance_toujours"]["classes_glitch"])))
    print("  %-7s | déclenchée      : %s" % (fam, ligne(RG[fam + "_redondance_auto"]["classes_glitch"])))
print("\n%-34s | %-26s | %s" % ("contre-mesure FIA", "redondance pendant l'attaque", "surcoût moyen en usage normal (en plus du bruit)"))
print("%-34s | %26s | %s" % ("aucune", "0 %", "+0 %"))
print("%-34s | %26s | %+.0f %% en permanence" % ("redondance toujours active (classique)", "100 %", CO["surcout_redondance_pct"]))
print("%-34s | %25.0f %% | %s" % ("NOTRE MODÈLE (arbre v12)", 100 * cov12,
      ", ".join("%s %+.1f %%" % (n, cout_usage(a)) for n, a in ALERTE.items())))

In [ ]:
# ==== Cellule 13.6 : synthèse — une figure SCA + FIA pour le mémoire et la soutenance (sans carte) ====
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for t in TAB_SCA:
    a1.scatter(t["surcout_pct"], t["effort_p10_8000"], color=coul[t["type"]], s=40, zorder=3)
    a1.annotate(t["contre_mesure"], (t["surcout_pct"], t["effort_p10_8000"]), fontsize=6, xytext=(4, 3), textcoords="offset points")
a1.axhline(32, color="0.4", ls="--", lw=0.8)
a1.set_xlabel("surcoût (%)"); a1.set_ylabel("effort restant à 8 000 traces (bits, 10 % pire cas)"); a1.set_title("SCA", fontsize=10)
pts = [("aucune", 0, 0, "0.5"), ("redondance toujours active", CO["surcout_redondance_pct"], 100, "tab:blue")] + \
      [("notre modèle : %s" % n, cout_usage(a), 100 * cov12, "tab:red") for n, a in ALERTE.items()]
for nom, x, y, c in pts:
    a2.scatter(x, y, color=c, s=40, zorder=3); a2.annotate(nom, (x, y), fontsize=6, xytext=(4, -9), textcoords="offset points")
a2.set_xlabel("surcoût moyen de la redondance en usage normal (%)"); a2.set_ylabel("redondance active pendant l'attaque (%)")
a2.set_title("FIA", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "synthese_comparaison_v12.pdf")); plt.show()

## 14. Attaque profilée par CNN : quelle protection résiste le mieux à l'apprentissage profond ?
Le MÊME attaquant CNN (même architecture, même budget de 30 000 traces d'apprentissage à clés aléatoires) est
entraîné sur CHAQUE contre-mesure classique et sur NOTRE MODÈLE déployé (politique v12 active), puis attaque la
campagne à clé fixe correspondante (sections 6, 8, 12, 13). Contrôle : l'AES sans bruit doit être cassé.
Méthode : Cagli et al. CHES 2017 (CNN + augmentation par décalage) ; Zaid et al. TCHES 2020 (architecture compacte).
Tout est écrit sur le disque de traces. Ordre : 14.1 (carte, ≈ 2 h) → 14.2 (arrière-plan, ≈ 3-4 h) → 14.3 → 14.4.

In [ ]:
import sca_capture as sc, banc_v10 as bv
importlib.reload(sc); importlib.reload(bv)

In [ ]:
# ==== Cellule 14.1 : caractérisation à CLÉS ALÉATOIRES, une par contre-mesure, 30 000 traces (≈ 2 h, reprise auto) ====
# Déjà faites : profil_sans_bruit_20k (contrôle) et profil_niveau3_50k (niveau 3 statique, 30 000 premières traces).
import politique_v10 as pol, fautes_v10 as fv, monitor_collect as mc
importlib.reload(pol); importlib.reload(fv)
CLS = json.load(open(os.path.join(bv.TRACE_ROOT, "classiques_v12.json")))["classiques"]
MELANGE = no.Config(((1, 3), (3, 3), (4, 3)), ((3, 0),), 0x403)        # même config que campagnes/melange_20k
class Politique:
    """« Configuration » = aucune : la politique v12 décide seule. Clé aléatoire à chaque trace -> budget neuf ->
    plancher calibré (≈ 4,8 % niveau 2, sinon niveau 3), exactement la distribution déployée."""
    def setup(self):
        def _s(nt_):
            sc.make_setup()(nt_)
            bv._lire_ligne(nt_, "H", [1], "H", 8)
            mc._cmd(nt_.target, "a", bytearray([1]))
        return _s
    def to_dict(self):
        return dict(comps=[], raw=[], mask=0, name="politique_v12 (arbre unique, active)")
PROFILS = {
    "profil_melange_30k": MELANGE,
    "profil_delais_aleatoires_30k": no.Config.from_dict(CLS["délais aléatoires"]["config"]),
    "profil_operations_factices_30k": no.Config.from_dict(CLS["opérations factices (type dual-rail)"]["config"]),
    "profil_bruit_alu_30k": no.Config.from_dict(CLS["bruit ALU"]["config"]),
    "profil_politique_v12_30k": Politique(),
}
fv.rapport_fautes(nt, mode=0, actif=False)                      # pas de redondance pendant la caractérisation
for k, (nom, cfg) in enumerate(PROFILS.items()):
    print("==", nom, "|", cfg.to_dict()["name"])
    bv.capturer_cles_aleatoires(nt, nom, 30000, cfg=cfg, seed=20 + k)
mc._cmd(nt.target, "a", bytearray([0]))
# contrôle : la politique a bien joué le plancher calibré (la durée révèle le niveau : ≈ 51 900 vs ≈ 57 900 cycles)
trg = np.load(os.path.join(bv.TRACE_ROOT, "apprentissage", "profil_politique_v12_30k", "trig.npy")).astype(float)
print("caractérisation de notre modèle : niveau 2 sur %.2f %% des traces (plancher attendu : 4,76 %%)"
      % (100 * np.mean(trg < 54900)))

In [ ]:
# ==== Cellule 14.2 : entraînement des 2 CNN puis attaques, en ARRIÈRE-PLAN (sans carte, ≈ 3-4 h) ====
# 1) CNN « sans bruit » -> attaque de sans_bruit_v10 (contrôle) ; 2) CNN « niveau 3 » -> attaques du niveau 3import glob
F = glob.glob(os.path.join(bv.TRACE_ROOT, "apprentissage", "profil_melange_30k", "chunk_*.npz"))
for f in F: os.remove(f)
print(len(F), "morceaux supprimés")
# statique (s0, s1, 50 000 traces) et de NOTRE MODÈLE déployé (politique_v12_20k). Reprise automatique.
import subprocess
CNN_DIR = os.path.join(bv.TRACE_ROOT, "cnn"); os.makedirs(CNN_DIR, exist_ok=True)
CMD = "%s cnn_v12.py tout > %s/log_cnn_v12.txt 2>&1" % (sys.executable, CNN_DIR)
proc_cnn = subprocess.Popen(["sh", "-c", CMD], cwd=bv.PROJET, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
                            start_new_session=True)
print("CNN lancé, PID", proc_cnn.pid, "| suivi :", os.path.join(CNN_DIR, "log_cnn_v12.txt"))
print("NE PAS relancer cette cellule tant que le journal n'affiche pas FINI.")

In [ ]:
print(open(os.path.join(bv.TRACE_ROOT, "cnn", "log_cnn_v12.txt")).read()[-800:])

In [ ]:
# ==== Cellule 14.3 : résultats — CNN contre CPA, pour chaque contre-mesure (quand le journal affiche FINI) ====
import matplotlib.pyplot as plt
import politique_v10 as pol
NIV, REF0 = pol.charger_niveaux()
CNN_DIR = os.path.join(bv.TRACE_ROOT, "cnn")
print(open(os.path.join(CNN_DIR, "log_cnn_v12.txt")).read()[-1200:])
RC = json.load(open(os.path.join(CNN_DIR, "resultats_cnn.json")))
V = os.path.join(bv.TRACE_ROOT, "validation"); C_ = os.path.join(bv.TRACE_ROOT, "campagnes")
CAND = "permutation@subbytes_pre:0+permutation@mixcolumns:0+jitter@addroundkey_pre:536+permutation@addroundkey_pre:0|m403"
RV = json.load(open(os.path.join(V, "resultats_validation.json")))
J = lambda *p: json.load(open(os.path.join(*p)))
# CPA : campagnes de 20 000 traces des sections 6, 8, 12 et 13 (même configuration ; contrôle, mélange et niveau 3 : v10)
CPA = {"aucune protection (contrôle)": J(C_, "sans_bruit_v10_resultat_w300.json"),
       "mélange seul": J(C_, "melange_20k_resultat_w300.json"),
       "délais aléatoires": J(V, "classiques", "delais_aleatoires_20k_resultat_w300.json"),
       "opérations factices (type dual-rail)": J(V, "classiques", "operations_factices_20k_resultat_w300.json"),
       "bruit ALU": J(V, "classiques", "bruit_alu_20k_resultat_w300.json"),
       "notre niveau 3 statique": RV["s0"][CAND],
       "NOTRE MODÈLE déployé (v12)": J(V, "politique_v12_20k_resultat_w300.json")}
def a8(r, cle="effort_p10"):
    return r[cle][r["ns"].index(8000)] if 8000 in r["ns"] else None
TAB = []
print("%-38s | %8s | %-26s | %-26s" % ("contre-mesure", "surcoût", "CPA A1-A4 : N(10 %) / bits", "CNN : N(10 %) / bits / octets"))
for nom, rc in RC.items():
    rp = CPA[nom]; s = 100 * (rp["trig_mean"] / REF0 - 1)
    if a8(rc) is None:
        continue
    TAB.append(dict(contre_mesure=nom, surcout_pct=s, N10_cpa=pol.n10(rp), bits_cpa=a8(rp), N10_cnn=rc["N10"],
                    bits_cnn=a8(rc), octets_cnn=a8(rc, "octets_trouves")))
    f = lambda n, r: str(n) if n else "> %d" % r["ns"][-1]
    print("%-38s | %+7.1f %% | %8s / %6.1f bits        | %8s / %6.1f bits / %4.1f"
          % (nom, s, f(pol.n10(rp), rp), a8(rp), f(rc["N10"], rc), a8(rc), a8(rc, "octets_trouves")))
json.dump(TAB, open(os.path.join(CNN_DIR, "comparaison_cnn_cpa.json"), "w"), indent=1)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
for t in TAB:
    nous = "notre" in t["contre_mesure"].lower() or "NOTRE" in t["contre_mesure"]
    ax.scatter(t["surcout_pct"], t["bits_cnn"], color="tab:red" if nous else ("0.5" if "aucune" in t["contre_mesure"] else "tab:blue"),
               marker="o", s=40, zorder=3)
    ax.annotate(t["contre_mesure"], (t["surcout_pct"], t["bits_cnn"]), fontsize=6.5, xytext=(4, 3), textcoords="offset points")
ax.axhline(32, color="0.4", ls="--", lw=0.8); ax.text(12, 34, "clé cassée (≤ 32 bits)", fontsize=7)
ax.set_xlabel("surcoût en cycles (%)"); ax.set_ylabel("effort restant face au CNN à 8 000 traces, 10 % pire cas (bits)")
ax.set_title("Attaque profilée par CNN : classiques (bleu) contre notre modèle (rouge)", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "cnn_comparaison.pdf")); plt.show()

In [ ]:
# ==== Cellule 14.4 : décision sur le masquage AES (règle fixée AVANT l'expérience) ====
# Règle : si le CNN casse notre modèle déployé ou le niveau 3 (10 % pire cas <= 32 bits) avec moins de 2 x 10^4 traces
# (marge m = 2 sur N_vie = 10^4), le masquage devient nécessaire ; sinon il passe en perspective.
# Préalable : le contrôle sans bruit doit être cassé, sinon le résultat n'est pas interprétable.
ctrl = RC["aucune protection (contrôle)"]["N10"]
print("contrôle sans bruit :", ("CASSÉ avec %d traces -> pipeline valide" % ctrl) if ctrl else "NON cassé -> résultat NON interprétable")
for nom in ("notre niveau 3 statique", "NOTRE MODÈLE déployé (v12)"):
    n = RC[nom]["N10"]
    print("%-28s : %s" % (nom, ("cassé avec %d traces -> MASQUAGE NÉCESSAIRE" % n) if (n and n < 20000)
                          else "non cassé jusqu'à %d traces -> masquage en perspective" % RC[nom]["ns"][-1]))

In [ ]:
# ==== Cellule 14.5 : campagnes d'attaque v12 (MÊME firmware que les profils) puis attaques CNN (carte, ≈ 30 min + 5 min) ====
# Le premier contrôle a échoué parce que les campagnes attaquées (sans bruit, mélange, niveau 3) dataient du firmware v10.
# On les recapture en v12, à clé fixe, 10 000 traces chacune ; les 7 CNN déjà entraînés sont réutilisés (aucun
# nouvel apprentissage). Ensuite : relancer 14.3 puis 14.4.
import subprocess, politique_v10 as pol, fautes_v10 as fv, sca_capture as sc
importlib.reload(sc); importlib.reload(bv); importlib.reload(pol); importlib.reload(fv)
NIV, REF0 = pol.charger_niveaux()
niv3 = [v for v in NIV if v["niveau"] == 3][0]
CFG3 = no.Config(pol.COMPS, ((1, niv3["d"]), (3, 0)), pol.MASK)
MELANGE = no.Config(((1, 3), (3, 3), (4, 3)), ((3, 0),), 0x403)
fv.rapport_fautes(nt, mode=0, actif=False)
for nom, cfg in (("sans_bruit_10k", None), ("melange_10k", MELANGE), ("niveau3_10k", CFG3)):
    print("==", nom)
    bv.capturer(nt, nom, 10000, cfg=cfg, sous_dossier=os.path.join("validation", "v12_attaque"), seed=7)
CNN_DIR = os.path.join(bv.TRACE_ROOT, "cnn")
r = subprocess.run([sys.executable, "cnn_v12.py", "tout"], cwd=bv.PROJET, capture_output=True, text=True)
open(os.path.join(CNN_DIR, "log_cnn_v12_attaques_v12.txt"), "w").write(r.stdout + r.stderr)
print("\n".join(l for l in r.stdout.splitlines() if "== attaque" in l or "n =  8000" in l or "FINI" in l))

## 15. Escalier calibré (firmware v13)
La v12 joue un **mélange constant** (4,76 % niveau 2, 95,24 % niveau 3) : le coût est le même, que la clé serve 100 fois
ou 10 000 fois. La v13 garde les mêmes niveaux, le même budget (limite 2^30/m, m = 2) et la même branche « faute » ;
seul le dernier nœud de l'arbre change : **niveau 2 tant que le budget consommé x1 reste bas, puis niveau 3**.
Le seuil est calculé par le planificateur en rejouant l'arithmétique entière de la carte, puis **distillé** dans l'arbre.
Garantie : la fuite cumulée reste sous 1/m pour toute durée de vie jusqu'à 10 000 chiffrements ; au-delà, alerte.

In [ ]:
# ==== Cellule 15.0 : planificateur, distillation de l'escalier, export C et contrôle C = Python (hors ligne, < 10 s) ====
import detecteur_v10 as dt, politique_v10 as pv
importlib.reload(dt); importlib.reload(pv)
NIV, REF0 = pv.charger_niveaux()
AU13 = dt.construire_arbre_v13(Jtout, NIV, n_vie=10000, m=2)          # sous-arbre des resets : identique à la v12
P13, E13 = AU13["plan"], AU13["escalier"]
print("1) PLANIFICATEUR : niveau %d tant que x1 <= %d ‰ -> %d chiffrements bon marché ; fuite à 10 000 = %.2f %% de la limite"
      % (P13["a"] + 1, P13["seuil"], P13["n_a"], 100 * P13["fuite_finale"]))
print("2) DISTILLATION (arbre de profondeur 2 sur x1) : marche si x1 <= %d ‰, épuisé si x1 > %d ‰ | accord %.2f %% (grille 0-1300 : %.2f %%)"
      % (E13["seuil_marche"], E13["seuil"], 100 * E13["accord"], 100 * E13["accord_grille"]))
print("   sous-arbre des resets identique à la v12 :",
      AU13["resets"].tree_.feature.tolist() == AU["resets"].tree_.feature.tolist()
      and AU13["resets"].tree_.threshold.tolist() == AU["resets"].tree_.threshold.tolist())
EXP13 = dt.export_politique_v13(AU13, NIV, chemin=os.path.join(os.getcwd(), "noise_policy_v13.h"),
                                commentaire="Entraîné sur 6 campagnes de glitch + 4 sessions normales (cellules 10.2, 10.2b, 11.1).")
V13 = dt.verifier_c_v13(AU13, EXP13, Jtout)
print("3) EXPORT C : %s" % EXP13["chemin"])
print("   contrôle C = Python : %d/%d états (niveau ET classe) | rejeu du budget sur %d chiffrements : %d/%d identiques"
      % (V13["etats_ok"], V13["etats"], V13["rejeu"], V13["rejeu_ok"], V13["rejeu"]))
print("   identique au noise_policy.h compilé dans le firmware :",
      open(EXP13["chemin"]).read() == open(os.path.join(FW_DIR, "noise_policy.h")).read())
print(dt.cout_selon_duree(NIV, P13).round(1).to_string(index=False))

In [ ]:
# ==== Cellule 15.1 : flashage du firmware v13 (carte branchée, ≈ 1 min) ====
import monitor_collect as mc, sessions_v10 as sv
for m_ in (bv, sc, no, pv, dt):
    importlib.reload(m_)
try:
    nt.scope.dis(); nt.target.dis()
except Exception:
    pass
nt = bv.connecter(programmer=True)                   # programme simpleserial-aes-CWLITEARM.hex (v13)
INFO = bv.info_firmware(nt)
assert INFO["version"] == 13, "la carte n'exécute pas le firmware v13"
BUILD_V13 = bv.empreinte_firmware()
print("build v13 :", BUILD_V13)

In [ ]:
# ==== Cellule 15.2 : non-régression v13 — AES correct et durées des niveaux identiques (≈ 2 min) ====
PT0 = np.frombuffer(bytes.fromhex("6bc1bee22e409f96e93d7e117393172a"), np.uint8)
CT0 = "3ad77bb40d7a3660a89ecaf32466ef97"
cfg_niv = lambda v: no.Config(pv.COMPS, ((1, v["d"]), (3, 0)), pv.MASK)
TESTS = [("générateur désactivé (0x000)", no.Config((), (), 0x000), REF0)] + \
        [("niveau %d (d = %d)" % (v["niveau"], v["d"]), cfg_niv(v), v["cycles"]) for v in NIV]
NONREG13 = []
for nom, cfg, ref in TESTS:
    mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
    bv.setup_v10(cfg)(nt)
    ok_aes = all(bytes(sc.capture_one(nt, PT0)[1]).hex() == CT0 for _ in range(10))
    t = bv.mesurer_durees(nt, cfg, n=200)
    ecart = 100 * (t.mean() / ref - 1)
    NONREG13.append(dict(test=nom, aes_ok=ok_aes, cycles=float(t.mean()), ref=ref, ecart_pct=ecart))
    print("%-30s AES %s | %7.0f cycles (réf. %7.0f) | écart %+5.2f %% %s"
          % (nom, "ok  " if ok_aes else "FAUX", t.mean(), ref, ecart, "" if abs(ecart) < 1 else "<-- À VÉRIFIER"))
json.dump(dict(build=BUILD_V13, tests=NONREG13), open(os.path.join(bv.TRACE_ROOT, "nonregression_v13.json"), "w"), indent=1)

In [ ]:
# ==== Cellule 15.2b : calibration de la durée nette du firmware v13 (≈ 1 min, lecture seule) ====
# Le détecteur de fautes compare la durée nette d'un AES (durée - délais insérés) à FAULT_NET_REF (±25 cycles).
# Cette référence dépend de la compilation : la v13 a décalé le code, d'où 47 543 (v12) -> 47 520 (v13).
# Attendu : écart médian proche de 0, aucune durée hors tolérance, compteur dur = 0.
import fautes_v10 as fv
FAULT_NET_REF, FAULT_NET_TOL = 47520, 25                  # valeurs compilées dans noise_generator.c (v13)
mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY)); mc._cmd(nt.target, "R", bytearray([0]))
bv._lire_ligne(nt, "H", [1], "H", 8)                      # compteurs de faute à zéro
mc._cmd(nt.target, "a", bytearray([1]))                   # politique v13
fv.rapport_fautes(nt, mode=0, actif=True)                 # lignes G (durée nette), redondance désactivée
rng = np.random.default_rng(5)
R = [fv.operation(nt, rng.integers(0, 256, 16, dtype=np.uint8)) for _ in range(400)]
fv.rapport_fautes(nt, mode=0, actif=False)
NET = np.array([r["g"]["net_cycles"] for r in R], float)
DP = [dt.lire_plog_v12(r["plog_hex"]) for r in R]
LVd = np.array([d["level"] for d in DP]); DURd = np.array([d["feats"][6] for d in DP])
print("référence compilée : %d ± %d cycles" % (FAULT_NET_REF, FAULT_NET_TOL))
for k in (1, 2):
    s = LVd == k
    if s.any():
        print("niveau %d : %3d chiffrements | durée nette médiane %.0f | min %.0f | max %.0f | écart médian %+.0f"
              % (k + 1, s.sum(), np.median(NET[s]), NET[s].min(), NET[s].max(), np.median(NET[s]) - FAULT_NET_REF))
hors = int((np.abs(NET - FAULT_NET_REF) > FAULT_NET_TOL).sum())
print("durées hors tolérance : %d / %d | compteur dur max : %d | classes : %s"
      % (hors, len(NET), DURd.max(), sorted({d["classe"] for d in DP})))
print("CALIBRATION", "OK" if hors == 0 and DURd.max() == 0 else "À REVOIR")
json.dump(dict(build=BUILD_V13, ref=FAULT_NET_REF, tol=FAULT_NET_TOL, net=NET.tolist(), niveaux=LVd.tolist()),
          open(os.path.join(bv.TRACE_ROOT, "calibration_net_v13.json"), "w"))

In [ ]:
# ==== Cellule 15.3 : escalier sur la carte — marches, budget, conformité, reset (≈ 9 min) ====
# Une clé neuve, 10 050 chiffrements : la carte doit jouer exactement 475 chiffrements au niveau 2, puis le niveau 3,
# puis passer en « budget épuisé » (classe 1) au chiffrement prévu. Un reset au chiffrement 300 (même clé) ne doit PAS
# faire redescendre l'escalier (budget en .noinit). Python rejoue l'arbre sur chaque décision : 0 erreur attendue.
import time
N13, RESET_A = 10050, 300
mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY))
mc._cmd(nt.target, "R", bytearray([0]))              # banc : budget de la clé remis à zéro (clé « neuve »)
bv._lire_ligne(nt, "H", [1], "H", 8)                 # banc : compteurs de faute à zéro
mc._cmd(nt.target, "a", bytearray([1]))              # politique embarquée (arbre v13)
rng = np.random.default_rng(13)
PL13, TRG13 = [], []
t0 = time.time()
for i in range(N13):
    if i == RESET_A:                                 # reset logiciel de la cible, même clé : le budget doit être conservé
        nt.reset(); time.sleep(0.3); mc._flush(nt.target)
        mc._cmd(nt.target, "k", bytearray(bv.KEY)); mc._cmd(nt.target, "a", bytearray([1]))
    _, _, trig, plog = sc.capture_one(nt, rng.integers(0, 256, 16, dtype=np.uint8), with_trig=True, with_plog=True)
    PL13.append(plog); TRG13.append(trig)
    if (i + 1) % 2000 == 0:
        print("  %5d / %d chiffrements (%.0f s)" % (i + 1, N13, time.time() - t0))
assert all(p is not None for p in PL13), "pas de journal 'P' : la politique n'est pas active"
D13 = [dt.lire_plog_v12(p) for p in PL13]
LV13, X1_13, CL13 = (np.array([d[k] for d in D13]) for k in ("level", "x1", "classe"))
TRG13 = np.array(TRG13, float)
# Rejeu enchaîné : Python recalcule le budget x1 de chaque chiffrement à partir des niveaux qu'il prédit lui-même
# (arithmétique entière de la carte) et des compteurs de faute lus sur la carte ; x1 ET niveau doivent coïncider.
L_, X1_PREVU, LV_PREVU = 0, [], []
for d in D13:
    x1 = min((L_ * 1000) // P13["limite"], 0xFFFF)
    lv_, _ = dt.decide_v13_py(AU13, np.array([x1]), np.array([d["feats"]]))
    X1_PREVU.append(x1); LV_PREVU.append(int(lv_[0])); L_ += P13["fuites"][int(lv_[0])]
X1_PREVU, LV_PREVU = np.array(X1_PREVU), np.array(LV_PREVU)
CONF13 = dt.conformite_v13(PL13, AU13)
i3 = int(np.argmax(LV13 == 2)); i_ep = int(np.argmax(CL13 == 1)) if (CL13 == 1).any() else None
print("marche 1 (niveau 2) : %d chiffrements (sans reset : %d) | premier niveau 3 (hors alerte) au chiffrement %d"
      % ((LV13 == 1).sum(), P13["n_a"], int(np.flatnonzero((LV13 == 2) & (CL13 == 0))[0]) + 1))
print("budget épuisé (classe 1) à partir du chiffrement %s | x1 avant/après le reset : %d / %d ‰ (doit continuer)"
      % (None if i_ep is None else i_ep + 1, X1_13[RESET_A - 1], X1_13[RESET_A]))
print("rejeu enchaîné du budget : x1 identique %d/%d | niveau identique %d/%d"
      % ((X1_13 == X1_PREVU).sum(), N13, (LV13 == LV_PREVU).sum(), N13))
print("classes : %s | alertes faute : %d (attendu : environ 13, juste après le reset)" %
      (sorted(set(CL13.tolist())), sum(d["alerte"] for d in D13)))
print("conformité v13 :", {k: v for k, v in CONF13.items() if k != "exemples"})
json.dump(dict(build=BUILD_V13, n=N13, reset_a=RESET_A, plog=[p if isinstance(p, str) else bytes(p).hex() for p in PL13],
               trig=TRG13.tolist()), open(os.path.join(bv.TRACE_ROOT, "escalier_v13.json"), "w"))

In [ ]:
# Diagnostic 15.3 (lecture seule)
import collections
print("niveaux joués :", dict(collections.Counter(LV13.tolist())))      # 1 = niveau 2, 2 = niveau 3
print("classes       :", dict(collections.Counter(CL13.tolist())))      # 0 escalier, 1 épuisé, 2 faute, 3 budget désactivé
print("x1 min / max  :", int(X1_13.min()), "/", int(X1_13.max()), "‰")
print("alertes faute :", sum(d["alerte"] for d in D13))
for i in (0, 1, 299, 300, 301, 315, 474, 475, 476, 480, 1000, 5000, 10049):
    print("  chiffrement %5d : niveau %d | classe %d | x1 %4d ‰ | compteurs %s"
          % (i + 1, LV13[i] + 1, CL13[i], X1_13[i], D13[i]["feats"]))
print("rejeu enchaîné : x1 %d/%d | niveau %d/%d" % ((X1_13 == X1_PREVU).sum(), N13, (LV13 == LV_PREVU).sum(), N13))
print("conformité :", {k: v for k, v in CONF13.items() if k != "exemples"})
print("exemples d'écart :", CONF13["exemples"][:2])

In [ ]:
# Diagnostic durée nette v13 (lecture seule, ≈ 1 min)
import fautes_v10 as fv
mc.prepare_target(nt.target); mc._cmd(nt.target, "k", bytearray(bv.KEY)); mc._cmd(nt.target, "R", bytearray([0]))
bv._lire_ligne(nt, "H", [1], "H", 8)                      # compteurs de faute à zéro
mc._cmd(nt.target, "a", bytearray([1]))                   # politique v13
fv.rapport_fautes(nt, mode=0, actif=True)                 # lignes G (durée nette), redondance désactivée
rng = np.random.default_rng(5)
R = [fv.operation(nt, rng.integers(0, 256, 16, dtype=np.uint8)) for _ in range(400)]
fv.rapport_fautes(nt, mode=0, actif=False)
NET = np.array([r["g"]["net_cycles"] for r in R], float)
LVd = np.array([dt.lire_plog_v12(r["plog_hex"])["level"] for r in R])
DURd = [dt.lire_plog_v12(r["plog_hex"])["feats"][6] for r in R]
print("référence firmware FAULT_NET_REF = 47543, tolérance ±25")
for k in (1, 2):
    s = LVd == k
    if s.any():
        print("niveau %d : %3d chiffrements | durée nette médiane %.0f | min %.0f | max %.0f | écart médian %+.0f"
              % (k + 1, s.sum(), np.median(NET[s]), NET[s].min(), NET[s].max(), np.median(NET[s]) - 47543))
print("5 premières durées nettes :", NET[:5].astype(int).tolist(), "| niveaux :", (LVd[:5] + 1).tolist())
print("compteur dur sur les 5 premières décisions :", DURd[:5])

In [ ]:
# ==== Cellule 15.4 : coût mesuré selon la durée de vie de la clé, v12 contre v13 (hors ligne) ====
# Coût d'une clé utilisée n fois = moyenne des durées mesurées des n premiers chiffrements / référence - 1.
# Les décisions en classe 2 (alerte après le reset) sont au niveau 3 : elles comptent dans le coût, comme en vrai.
cum = np.cumsum(TRG13) / np.arange(1, N13 + 1)
DUREES = [50, 100, 300, 475, 1000, 2000, 3000, 5000, 10000]
TAB = dt.cout_selon_duree(NIV, P13, DUREES)
TAB["escalier_v13_mesure"] = [100 * (cum[n - 1] / REF0 - 1) for n in DUREES]
print(TAB.round(1).to_string(index=False))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
kk, ESC = dt.simulation_escalier(NIV, n_vie=10000, m=2, p16=P13["base"]["p16"], a=P13["a"], b=P13["b"])
L13 = np.cumsum([P13["fuites"][l] for l in LV13[:10000]]) / (2 * P13["limite"])     # niveaux réellement joués
ax1.plot(kk, ESC["mélange constant"]["L"], label="mélange constant (v12)")
ax1.plot(kk, L13, label="escalier calibré (v13, mesuré)")
ax1.plot(kk, ESC["escalier glouton (v9)"]["L"], ls=":", label="escalier glouton (v9)")
ax1.axhline(0.5, color="0.4", ls="--", lw=0.8); ax1.text(9800, 0.45, "limite 1/m (m = 2)", fontsize=8, ha="right")
ax1.set_xlabel("chiffrements depuis le chargement de la clé"); ax1.set_ylabel("fuite cumulée  Σ 1/N")
ax1.set_title("(a) budget de fuite", loc="left", fontsize=9); ax1.legend(fontsize=7, frameon=False)
n_ax = np.arange(1, 10001)
ax2.plot(n_ax, np.full(10000, TAB["melange_v12"].iloc[0]), label="mélange constant v12 (prévu)")
ax2.plot(n_ax, [(min(n, P13["n_a"]) * NIV[P13["a"]]["surcout"] + max(n - P13["n_a"], 0) * NIV[P13["b"]]["surcout"]) / n
                for n in n_ax], label="escalier v13 (prévu)")
ax2.plot(DUREES, TAB["escalier_v13_mesure"], "o", ms=4, label="escalier v13 (mesuré sur la carte)")
ax2.set_xscale("log"); ax2.set_xlabel("durée de vie réelle de la clé (chiffrements)"); ax2.set_ylabel("surcoût moyen (%)")
ax2.set_title("(b) coût selon l'usage de la clé", loc="left", fontsize=9); ax2.legend(fontsize=7, frameon=False)
for a_ in (ax1, ax2):
    a_.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(bv.TRACE_ROOT, "escalier_v13.pdf")); plt.show()

In [ ]:
# ==== Cellule 16.0 : réglages communs des figures (hors ligne) ====
import os, json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sca_eval import SBOX, HW                       # S-box AES et poids de Hamming (module du projet)

TR = bv.TRACE_ROOT                                   # .../trace/v10
FIG = os.environ.get("FIG_MEMOIRE", os.path.join(TR, "figures_memoire")); os.makedirs(FIG, exist_ok=True)
KEY0 = np.frombuffer(bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c"), np.uint8)
REF0 = 40262                                         # cycles, AES avec le générateur désactivé (masque 0x000)
NOTRE, CLASSIQUE, GRIS = "#1f5fa8", "#b5651d", "0.55"   # bleu = notre générateur, brun = classiques
mpl.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "pdf.fonttype": 42,
                     "savefig.bbox": "tight", "savefig.dpi": 300})
V3 = ("validation/permutation_subbytes_pre0__permutation_mixcolumns0__jitter_addroundkey_pre%d"
      "__permutation_addroundkey_pre0_m403")


def charger(rel, n=None):
    """traces (int16, lues à la demande), textes clairs, durées et méta d'une campagne."""
    d = os.path.join(TR, rel)
    X = np.load(os.path.join(d, "traces.npy"), mmap_mode="r")
    P = np.load(os.path.join(d, "pts.npy")); t = np.load(os.path.join(d, "trig.npy"))
    m = json.load(open(os.path.join(d, "meta.json")))
    n = len(P) if n is None else min(n, len(P))
    return X[:n], P[:n], t[:n], m


def corr_bonne_cle(X, P, octet=0, cle=KEY0, chunk=1000):
    """Corrélation de Pearson entre HW(S(p ^ k)) (bonne clé) et chaque échantillon de la trace."""
    h = HW[SBOX[P[:, octet] ^ cle[octet]]].astype(np.float64)
    n = len(h); sh, sh2 = h.sum(), (h ** 2).sum()
    st = np.zeros(X.shape[1]); st2 = np.zeros(X.shape[1]); sht = np.zeros(X.shape[1])
    for s in range(0, n, chunk):
        x = np.asarray(X[s:s + chunk], dtype=np.float64)
        st += x.sum(0); st2 += (x ** 2).sum(0); sht += h[s:s + chunk] @ x
    return (n * sht - sh * st) / np.sqrt((n * sh2 - sh ** 2) * np.maximum(n * st2 - st ** 2, 1e-12))


def sauver(fig, nom):
    p = os.path.join(FIG, nom); fig.savefig(p); fig.savefig(p.replace(".pdf", ".png")); print("écrit :", p)

In [ ]:
# ==== Cellule 16.1 : Figure 5.2 — carte de fuite (SNR) de la cible non protégée (hors ligne, ≈ 1 min) ====
# SNR(t) = variance des moyennes par classe / moyenne des variances par classe ; classe = HW(S(p ^ k)).
X, P, _, m = charger("apprentissage/profil_sans_bruit_20k")
K = np.load(os.path.join(TR, "apprentissage/profil_sans_bruit_20k/keys.npy"))   # une clé par trace
OCTETS = (0, 5, 10, 15)
fig, ax = plt.subplots(figsize=(6.3, 2.4))
for i, b in enumerate(OCTETS):
    cl = HW[SBOX[P[:, b] ^ K[:, b]]].astype(int)
    s1 = np.zeros((9, X.shape[1])); s2 = np.zeros((9, X.shape[1])); nc = np.bincount(cl, minlength=9).astype(float)
    for s in range(0, len(cl), 2000):
        x = np.asarray(X[s:s + 2000], dtype=np.float64); c = cl[s:s + 2000]
        for v in np.unique(c):
            xv = x[c == v]; s1[v] += xv.sum(0); s2[v] += (xv ** 2).sum(0)
    ok = nc > 1
    mu = s1[ok] / nc[ok, None]; var = s2[ok] / nc[ok, None] - mu ** 2
    w = nc[ok] / nc[ok].sum()
    signal = (w[:, None] * (mu - (w[:, None] * mu).sum(0)) ** 2).sum(0)
    snr = signal / np.maximum((w[:, None] * var).sum(0), 1e-12)
    ax.plot(snr, lw=0.6, color=plt.cm.Blues(0.45 + 0.15 * i), label="octet %d" % b)
    print("octet %2d : SNR max %.2f à l'échantillon %d" % (b, snr.max(), snr.argmax()))
ax.set_xlim(0, 6000); ax.set_xlabel("échantillon (1 par cycle d'horloge)"); ax.set_ylabel("SNR")
ax.set_title("La fuite de la sortie de la S-box se concentre au début de la trace", loc="left")
ax.legend(ncol=1, loc="center right")
fig.text(0.0, -0.1, "n = %d traces, clés aléatoires, firmware v%s ; classes : poids de Hamming de S(p ⊕ k). "
         "Échantillons 0 à 6 000 sur 24 400 : le reste de la trace ne montre aucune fuite."
         % (len(P), m.get("firmware")), fontsize=6, color="0.4")
sauver(fig, "carte_snr.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.2 : Figure 5.5 — délais seuls, avant et après réalignement par A3 (hors ligne, ≈ 30 s) ====
# Campagne « délais avant SubBytes seuls » (au plus 10 itérations) : le délai décale d'un bloc toute la suite du calcul ; le réalignement
# sur un motif de 60 échantillons pris juste avant la S-box visée le compense entièrement.
import acceleration
X, P, _, m = charger("validation/jitter_subbytes_pre10_m003", n=3000)
A0, A1_ = 2300, 2900                                   # fenêtre affichée
MS = 1100                                              # décalage maximal cherché
R, lags = acceleration.resync_fast(np.asarray(X), ref_start=2400, ref_len=60, max_shift=MS)
rb = corr_bonne_cle(np.asarray(X[:, A0:A1_]), P)
ra = corr_bonne_cle(R[:, A0 - MS:A1_ - MS], P)         # resync_fast rogne MS échantillons au début
fig, axs = plt.subplots(2, 2, figsize=(6.3, 3.6), sharex=True, gridspec_kw=dict(height_ratios=[1.4, 1]))
t = np.arange(A0, A1_)
for j, (Y, r, tit) in enumerate(((np.asarray(X[:12, A0:A1_]), rb, "(a) Avant réalignement"),
                                 (R[:12, A0 - MS:A1_ - MS], ra, "(b) Après réalignement (attaquant A3)"))):
    for y in Y:
        axs[0, j].plot(t, y / 1024 - 0.5, lw=0.4, alpha=0.7, color=CLASSIQUE)
    axs[0, j].set_title(tit, loc="left")
    axs[1, j].plot(t, r, lw=0.7, color="k"); axs[1, j].set_ylim(-1, 1)
    axs[1, j].set_xlabel("échantillon")
    axs[1, j].text(0.02, 0.08, "|ρ| max = %.2f" % np.abs(r).max(), transform=axs[1, j].transAxes, fontsize=7)
axs[0, 0].set_ylabel("trace (unité ADC)"); axs[1, 0].set_ylabel("corrélation\nde la bonne clé")
fig.tight_layout()
fig.text(0.0, -0.03, "12 traces affichées ; corrélation sur %d traces (octet 0, poids de Hamming de la sortie de "
         "S-box)." % len(P), fontsize=6, color="0.4")
sauver(fig, "realignement.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.3 : Figure 5.4 — surcoût de chaque générateur selon son intensité (CARTE, ≈ 4 min) ====
# Mesure sur la carte (mode forcé, politique coupée), masque des rondes 0x403, 100 chiffrements par point.
GRILLE = {  # générateur : (zone utilisée, numéro du générateur, intensités du tableau 4.1)
    "délais aléatoires (zone 1)":    (1, 1, (10, 25, 50, 100, 200, 400, 800)),
    "double rail logiciel (zone 2)": (2, 2, (3, 6, 12, 25, 50, 100)),
    "mélange (zones 1 et 3)":        (None, 3, (0, 1, 2, 4)),
    "bruit arithmétique (zone 3)":   (3, 4, (5, 20, 50, 200)),
}
ref = float(bv.mesurer_durees(nt, no.Config((), (), 0x000), n=100).mean())
print("référence mesurée : %.0f cycles (attendu ≈ %d)" % (ref, REF0))
CAL = {}
for nom, (z, g, grille) in GRILLE.items():
    pts = []
    for v in grille:
        comps = ((1, 3), (3, 3)) if g == 3 else ((z, g),)
        d = bv.mesurer_durees(nt, no.Config(comps, ((g, v),), 0x403), n=100)
        pts.append((v, 100 * (d.mean() / ref - 1)))
        print("  %-30s intensité %4d : %+6.1f %%" % (nom, v, pts[-1][1]))
    CAL[nom] = pts
json.dump(dict(ref=ref, points=CAL), open(os.path.join(FIG, "calibration_generateurs.json"), "w"), indent=1)
fig, axs = plt.subplots(1, 4, figsize=(6.6, 2.0), sharey=True)
for ax, (nom, pts) in zip(axs, CAL.items()):
    v, c = zip(*pts)
    ax.plot(v, c, "o-", ms=3, lw=1, color=CLASSIQUE)
    ax.set_title(nom.split(" (")[0], loc="left"); ax.set_xlabel("intensité")
    if nom.startswith(("délais", "double")):
        ax.set_xscale("log")
axs[0].set_ylabel("surcoût (%)")
fig.tight_layout()
fig.text(0.0, -0.1, "Mesuré sur la carte, 100 chiffrements par point, rondes 0, 1 et 10 ; référence : générateur "
         "désactivé.", fontsize=6, color="0.4")
sauver(fig, "calibration.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.4 : Figure 5.6 — configurations évaluées par ParEGO et front de Pareto (hors ligne) ====
H = json.load(open(os.path.join(TR, "parego", "opt11", "history_moy.json")))
BASE_PE = 45045.0                                    # référence utilisée par ParEGO (masque 0x7FF)
cout = np.array([100 * (BASE_PE * (1 + h["overhead_pct"] / 100) / REF0 - 1) for h in H])   # rebasé sur 40 262
lg = np.array([h["log10_n"] for h in H]); cens = np.array([bool(h.get("censored")) for h in H])
how = np.array([h.get("how", "") for h in H])
front = [i for i in range(len(H)) if not any((cout[j] <= cout[i]) and (lg[j] >= lg[i]) and
                                             ((cout[j] < cout[i]) or (lg[j] > lg[i])) for j in range(len(H)))]
front = sorted(front, key=lambda i: cout[i])
fig, ax = plt.subplots(figsize=(4.6, 3.0))
ax.scatter(cout[how == "seed"], lg[how == "seed"], s=16, facecolor="white", edgecolor=GRIS, label="plan initial")
ax.scatter(cout[how != "seed"], lg[how != "seed"], s=16, color=GRIS, label="proposées par ParEGO")
ax.plot(cout[front], lg[front], "-o", ms=4, lw=1, color=NOTRE, label="front de Pareto")
if cens.any():
    ax.scatter(cout[cens], lg[cens], s=40, marker="^", facecolor="none", edgecolor="k", lw=0.6,
               label="borne inférieure (non cassé)")
ax.set_xlabel("surcoût (%)"); ax.set_ylabel("log₁₀ N (traces, formule de Mangard)")
ax.set_title("ParEGO trouve les compromis coût / sécurité en %d mesures" % len(H), loc="left")
ax.legend(loc="lower right")
fig.text(0.0, -0.06, "%d configurations, 6 000 traces chacune ; N déduit de ρ* du meilleur attaquant (A1 à A4)."
         % len(H), fontsize=6, color="0.4")
sauver(fig, "pareto_reel.pdf"); plt.show()

In [ ]:
import glob
for p in sorted(glob.glob(os.path.join(TR, "**", "*.json"), recursive=True)):
    try:
        d = json.load(open(p))
    except Exception:
        continue
    # cherche récursivement un dict avec "ns" et "effort_med"
    def a_courbes(x, prof=0):
        if prof > 3: return False
        if isinstance(x, dict):
            if "ns" in x and "effort_med" in x: return True
            return any(a_courbes(v, prof + 1) for v in x.values())
        if isinstance(x, list):
            return any(a_courbes(v, prof + 1) for v in x[:5])
        return False
    if a_courbes(d):
        print(os.path.relpath(p, TR))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))
cmap = plt.get_cmap("viridis")
for i, nom in enumerate(noms):
    col = cmap(i / (len(noms) - 1) * 0.9)
    for s, ls in (("s0", "-"), ("s1", "--")):
        r = COURBES[nom].get(s)
        if r is None: continue
        ax.plot(r["ns"], r["effort_med"], ls=ls, marker="o", ms=2.5, color=col,
                label="%s (%+.0f %%)" % (nom, SURCOUT[nom]) if s == "s0" else None)
# ... puis le même habillage que la cellule 8.4

In [ ]:
# ---- Figure 6.1 : effort restant selon le nombre de traces, toutes contre-mesures (style cellule 8.4) ----
C = json.load(open(os.path.join(TR, "comparaison_sca_v12.json")))
COUT = {c["contre_mesure"]: c["surcout_pct"] for c in C}

def courbes(path):
    """Renvoie toutes les courbes {ns, effort_med} trouvées dans un JSON (ordre d'apparition)."""
    out = []
    def rec(x):
        if isinstance(x, dict):
            if "ns" in x and "effort_med" in x: out.append(x); return
            for v in x.values(): rec(v)
        elif isinstance(x, list):
            for v in x: rec(v)
    rec(json.load(open(os.path.join(TR, path))))
    return out

# nom lisible -> (clé de comparaison_sca_v12.json, fichier(s) de courbes)
SRC = {
    "sans protection":      ("aucune protection",                     ["campagnes/sans_bruit_v10_resultat_w300.json"]),
    "mélange seul":         ("mélange seul (shuffling)",              ["campagnes/melange_20k_resultat_w300.json"]),
    "délais aléatoires":    ("délais aléatoires",                     ["validation/classiques/delais_aleatoires_20k_resultat_w300.json"]),
    "opérations factices":  ("opérations factices (type dual-rail)",  ["validation/classiques/operations_factices_20k_resultat_w300.json"]),
    "bruit arithmétique":   ("bruit ALU",                             ["validation/classiques/bruit_alu_20k_resultat_w300.json"]),
    "modèle déployé":       ("NOTRE MODÈLE déployé (arbre v12)",      ["validation/politique_v12_20k_resultat_w300.json"]),
}
# niveau 3 : une configuration de resultats_validation.json (à choisir parmi les clés de RV["s0"])
RV = json.load(open(os.path.join(TR, "validation", "resultats_validation.json")))
N3 = "permutation@subbytes_pre:0+jitter@mixcolumns:25+permutation@mixcolumns:0+permutation@addroundkey_pre:0|m403"  # <- à adapter
CURVES = {}
for nom, (cle, fichiers) in SRC.items():
    cs = [c for f in fichiers for c in courbes(f)]
    CURVES[nom] = (cs, COUT[cle])
    print("%-22s %d courbe(s)  surcoût %+.1f %%" % (nom, len(cs), COUT[cle]))
CURVES["niveau 3"] = ([RV["s0"][N3], RV["s1"][N3]], COUT["notre niveau 3 statique (ParEGO), s0"])

noms = sorted(CURVES, key=lambda n: CURVES[n][1])
COL = {   # une couleur fixe par configuration (les nôtres en bleu/vert, les classiques en tons chauds/neutres)
    "sans protection":     "#000000",
    "mélange seul":        "#E69F00",
    "délais aléatoires":   "#D55E00",
    "opérations factices": "#CC79A7",
    "bruit arithmétique":  "#7F7F7F",
    "niveau 3":            "#56B4E9",
    "modèle déployé":      "#0072B2",
}
MRK = {"sans protection": "o", "mélange seul": "s", "délais aléatoires": "^", "opérations factices": "D",
       "bruit arithmétique": "v", "niveau 3": "P", "modèle déployé": "*"}

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for n in noms:
    cs, cout = CURVES[n]
    for j, r in enumerate(cs[:2]):
        ax.plot(r["ns"], r["effort_med"], ls="-" if j == 0 else "--", lw=1.6,
                marker=MRK[n], ms=4.5 if MRK[n] != "*" else 7, color=COL[n],
                label="%s (%+.0f %%)" % (n, cout) if j == 0 else None)
ax.axhline(32, ls=":", c="0.4")
ax.text(8000, 34, "seuil : 32 bits restants (clé retrouvée en dessous)", fontsize=7, color="0.35", ha="right")
ax.set_xscale("log"); ax.set_ylim(-3, 110)
ax.set_xlabel("traces dont dispose l'attaquant (meilleur des attaquants A1 à A4)")
ax.set_ylabel("effort restant médian (bits)")
ax.set_title("Effort restant de l'attaquant selon le nombre de traces)",
             loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
          title="(surcoût par rapport au générateur désactivé)", title_fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(TR, "cout_securite_v12.pdf")); plt.show()

In [ ]:
# ==== Cellule 16.5 : Figure 6.1 — effort restant de l'attaquant selon le surcoût (hors ligne) ====
C = json.load(open(os.path.join(TR, "comparaison_sca_v12.json")))
NOMS = {"aucune protection": "sans protection", "mélange seul (shuffling)": "mélange seul",
        "délais aléatoires": "délais aléatoires", "opérations factices (type dual-rail)": "opérations factices",
        "bruit ALU": "bruit arithmétique", "notre niveau 3 statique (ParEGO), s0": "niveau 3 (campagne 1)",
        "notre niveau 3 statique (ParEGO), s1": "niveau 3 (campagne 2)",
        "NOTRE MODÈLE déployé (arbre v12)": "modèle déployé"}
POS = {"sans protection": (0, 10, "center"), "mélange seul": (0, 4, "center"),
       "délais aléatoires": (-0.6, 0, "right"), "opérations factices": (-0.6, 11, "right"),
       "bruit arithmétique": (-0.6, 4, "right"), "niveau 3 (campagne 1)": (-0.6, 2, "right"),
       "niveau 3 (campagne 2)": (-0.6, -2, "right"), "modèle déployé": (-0.6, 0, "right")}
fig, ax = plt.subplots(figsize=(4.8, 3.0))
ax.axhspan(-4, 32, color="0.94", lw=0); ax.text(9, 30, "clé cassée (< 32 bits)", fontsize=6, color="0.4", va="top")
for c in C:
    nom = NOMS[c["contre_mesure"]]; dx, dy, ha = POS[nom]
    ax.scatter(c["surcout_pct"], c["effort_p10_8000"], s=28, zorder=3,
               color=NOTRE if c["type"] == "nôtre" else CLASSIQUE)
    ax.text(c["surcout_pct"] + dx, c["effort_p10_8000"] + dy, nom, fontsize=7, ha=ha, va="center")
ax.set_xlim(8, 47); ax.set_ylim(-4, 100)
ax.set_xlabel("surcoût (%)"); ax.set_ylabel("effort restant à 8 000 traces (bits)")
ax.set_title("À coût égal (+43 %), notre générateur laisse plus du double d'effort", loc="left")
fig.text(0.0, -0.09, "Bleu : notre générateur ; brun : contre-mesures classiques.\nEffort : 10 % des tirages les plus "
         "favorables à l'attaquant (A1 à A4), 20 000 traces par condition.", fontsize=6, color="0.4")
sauver(fig, "cout_securite_v12.pdf"); plt.show()

In [ ]:
# ---- Figure 6.1 : effort restant selon le nombre de traces (une courbe par entrée de comparaison_sca_v12.json) ----
C = json.load(open(os.path.join(TR, "comparaison_sca_v12.json")))
NOMS = {"aucune protection": "sans protection", "mélange seul (shuffling)": "mélange seul",
        "délais aléatoires": "délais aléatoires", "opérations factices (type dual-rail)": "opérations factices",
        "bruit ALU": "bruit arithmétique", "notre niveau 3 statique (ParEGO), s0": "niveau 3 (campagne 1)",
        "notre niveau 3 statique (ParEGO), s1": "niveau 3 (campagne 2)",
        "NOTRE MODÈLE déployé (arbre v12)": "modèle déployé"}

def courbes(path):
    out = []
    def rec(x):
        if isinstance(x, dict):
            if "ns" in x and "effort_med" in x: out.append(x); return
            for v in x.values(): rec(v)
        elif isinstance(x, list):
            for v in x: rec(v)
    rec(json.load(open(os.path.join(TR, path))))
    return out

FICHIER = {   # entrée de C -> fichier de courbes (None = à retrouver dans RV)
    "aucune protection":                    "campagnes/sans_bruit_v10_resultat_w300.json",
    "mélange seul (shuffling)":             "campagnes/melange_20k_resultat_w300.json",
    "délais aléatoires":                    "validation/classiques/delais_aleatoires_20k_resultat_w300.json",
    "opérations factices (type dual-rail)": "validation/classiques/operations_factices_20k_resultat_w300.json",
    "bruit ALU":                            "validation/classiques/bruit_alu_20k_resultat_w300.json",
    "NOTRE MODÈLE déployé (arbre v12)":     "validation/politique_v12_20k_resultat_w300.json",
    "notre niveau 3 statique (ParEGO), s0": None,
    "notre niveau 3 statique (ParEGO), s1": None,
}
RV = json.load(open(os.path.join(TR, "validation", "resultats_validation.json")))

def e8000(r):
    return r["effort_med"][int(np.argmin(np.abs(np.array(r["ns"]) - 8000)))]

def courbe_de(c):
    f = FICHIER[c["contre_mesure"]]
    if f is not None:
        return courbes(f)[0]
    seed = "s0" if c["contre_mesure"].endswith("s0") else "s1"
    ecarts = sorted((abs(e8000(RV[seed][n]) - c["effort_med_8000"]), n) for n in RV[seed])
    print("%s -> %s (écart %.2f bit)" % (NOMS[c["contre_mesure"]], ecarts[0][1], ecarts[0][0]))
    return RV[seed][ecarts[0][1]]

COL = {"sans protection": "#000000", "mélange seul": "#E69F00", "délais aléatoires": "#D55E00",
       "opérations factices": "#CC79A7", "bruit arithmétique": "#7F7F7F",
       "niveau 3": "#56B4E9", "modèle déployé": "#0072B2"}
MRK = {"sans protection": "o", "mélange seul": "s", "délais aléatoires": "^", "opérations factices": "D",
       "bruit arithmétique": "v", "niveau 3": "P", "modèle déployé": "*"}

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for c in sorted(C, key=lambda c: c["surcout_pct"]):
    nom = NOMS[c["contre_mesure"]]
    base = nom.replace(" (campagne 1)", "").replace(" (campagne 2)", "")
    camp2 = nom.endswith("(campagne 2)")
    r = courbe_de(c)
    ax.plot(r["ns"], r["effort_med"], ls="--" if camp2 else "-", lw=1.6,
            marker=MRK[base], ms=7 if MRK[base] == "*" else 4.5, color=COL[base],
            label=None if camp2 else "%s (%+.0f %%)" % (base, c["surcout_pct"]))

ax.axhline(32, ls=":", c="0.4")
ax.text(8000, 34, "seuil : 32 bits restants (clé retrouvée en dessous)", fontsize=7, color="0.35", ha="right")
ax.set_xscale("log"); ax.set_ylim(-3, 110)
ax.set_xlabel("traces dont dispose l'attaquant (meilleur des attaquants A1 à A4)")
ax.set_ylabel("effort restant médian (bits)")
ax.set_title("Effort restant de l'attaquant selon le nombre de traces",
             loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
          title="(surcoût par rapport au générateur désactivé)", title_fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(TR, "cout_securite_v12.pdf")); plt.show()

In [ ]:
# ---- Figure 6.1 (variante courbes) : effort restant selon le surcoût, un tracé par budget de traces ----
BUDGETS = [500, 1000, 3000, 8000]
COLB = {500: "#D55E00", 1000: "#E69F00", 3000: "#009E73", 8000: "#0072B2"}   # bien distincts

def effort_a(cs, N):
    """Effort médian à N traces (valeur du ns le plus proche), moyenné sur les campagnes disponibles."""
    v = []
    for r in cs[:2]:
        k = int(np.argmin(np.abs(np.array(r["ns"]) - N)))
        v.append(r["effort_med"][k])
    return float(np.mean(v))

ordre = sorted(CURVES, key=lambda n: CURVES[n][1])
xs = [CURVES[n][1] for n in ordre]

fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.axhspan(-3, 32, color="0.94", lw=0)
ax.axhline(32, ls=":", c="0.4")
ax.text(max(xs) + 1, 34, "seuil : 32 bits restants (clé retrouvée en dessous)",
        fontsize=7, color="0.35", ha="right")

for N in BUDGETS:
    ys = [effort_a(CURVES[n][0], N) for n in ordre]
    ax.plot(xs, ys, "-o", color=COLB[N], lw=1.6, ms=4, label="%s traces" % format(N, ",").replace(",", " "))

# étiquettes des configurations sur la courbe du plus grand budget
ys_last = [effort_a(CURVES[n][0], BUDGETS[-1]) for n in ordre]
for n, x, y in zip(ordre, xs, ys_last):
    ax.annotate(n, (x, y), xytext=(0, -9), textcoords="offset points",
                fontsize=6.5, ha="center", color="0.25", rotation=0)

ax.set_xlim(min(xs) - 3, max(xs) + 3); ax.set_ylim(-3, 110)
ax.set_xlabel("surcoût par rapport au générateur désactivé (%)")
ax.set_ylabel("effort restant médian (bits)")
ax.set_title("Effort restant de l'attaquant selon le surcoût, pour plusieurs budgets de traces",
             loc="left", fontsize=9)
ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=4,
          title="traces dont dispose l'attaquant (meilleur des attaquants A1 à A4)", title_fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(TR, "cout_securite_v12_surcout.pdf")); plt.show()

In [ ]:
# ==== Cellule 16.6 : Figure 6.2 — corrélation de la bonne clé le long de la trace (hors ligne, ≈ 1 min) ====
CAS = (("sans protection", "validation/v12_attaque/sans_bruit_10k", GRIS),
       ("délais aléatoires (+43 %)", "validation/classiques/delais_aleatoires_20k", CLASSIQUE),
       ("niveau 3 (+43 %)", V3 % 536, NOTRE))
fig, axs = plt.subplots(3, 1, figsize=(6.3, 3.8), sharex=True, sharey=True)
for ax, (nom, rel, col) in zip(axs, CAS):
    X, P, _, _ = charger(rel, n=5000)
    r = corr_bonne_cle(X, P, octet=0)
    ax.plot(r, lw=0.5, color=col); ax.set_ylim(-1, 1)
    ax.text(0.99, 0.82, "%s : |ρ| max = %.2f" % (nom, np.abs(r[300:]).max()), transform=ax.transAxes,
            ha="right", fontsize=7)
axs[1].set_ylabel("corrélation de la bonne clé (octet 0)"); axs[-1].set_xlabel("échantillon"); axs[-1].set_xlim(0, 6000)
axs[0].set_title("Sans réalignement, les délais étalent le pic ; le mélange le répartit entre 16 créneaux", loc="left")
fig.tight_layout()
fig.text(0.0, -0.03, "5 000 traces par condition, poids de Hamming de la sortie de S-box, sans réalignement ; échantillons 0 à 6 000.",
         fontsize=6, color="0.4")
sauver(fig, "correlation_trace.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.7 : Figure 6.3 — escalier calibré mesuré sur la carte (hors ligne) ====
J = json.load(open(os.path.join(TR, "escalier_v13.json")))
pl = [bytes.fromhex(p) for p in J["plog"]]
LV = np.array([b[0] for b in pl])                                 # niveau joué (0, 1, 2 = niveaux 1, 2, 3)
FUITE = np.array([894785, 268435, 42950]) / 2.0 ** 30             # 1/N par niveau (table du firmware)
L13 = np.cumsum(FUITE[LV])                                         # fuite cumulée des niveaux réellement joués
NV = 10000
L12 = np.cumsum(FUITE[np.where(np.random.default_rng(0).random(NV) < 3121 / 65536, 1, 2)])  # mélange v12
lv9, s = [], 0.0                                                    # escalier glouton v9
for _ in range(NV):
    l = 0 if s < 0.5 else 2; lv9.append(l); s += FUITE[l]
L9 = np.cumsum(FUITE[lv9])
t13 = np.asarray(J["trig"], float)
t12 = np.load(os.path.join(TR, "validation/politique_v12_20k/trig.npy")).astype(float)
nn = np.arange(1, NV + 1)
c13 = 100 * (np.cumsum(t13[:NV]) / nn / REF0 - 1); c12 = 100 * (np.cumsum(t12[:NV]) / nn / REF0 - 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.7))
a1.plot(nn, L12, color=GRIS, lw=1, label="mélange constant (v12)")
a1.plot(np.arange(1, len(L13) + 1), L13, color=NOTRE, lw=1.4, label="escalier calibré (v13, mesuré)")
a1.plot(nn, L9, color=CLASSIQUE, lw=1, ls=":", label="escalier glouton (v9)")
a1.axhline(0.5, color="0.4", lw=0.7, ls="--")
a1.text(3000, 0.52, "limite 1/m (m = 2)", fontsize=6, ha="left", va="bottom")
a1.set_xlabel("chiffrements depuis le chargement de la clé"); a1.set_ylabel("fuite cumulée Σ 1/N")
a1.set_title("(a) La garantie tient jusqu'à 10 000 chiffrements", loc="left"); a1.legend(loc="upper left")
a2.plot(nn, c12, color=GRIS, lw=1, label="mélange constant (v12, mesuré)")
a2.plot(nn, c13, color=NOTRE, lw=1.4, label="escalier calibré (v13, mesuré)")
a2.set_xscale("log"); a2.set_xlim(10, NV)
a2.set_xlabel("durée de vie réelle de la clé (chiffrements)"); a2.set_ylabel("surcoût moyen (%)")
a2.set_title("(b) Le coût suit l'usage de la clé", loc="left"); a2.legend(loc="lower right")
fig.tight_layout()
fig.text(0.0, -0.06, "v13 : 10 050 chiffrements sur la carte, redémarrage au 300e (%d au niveau 2) ; v12 : 20 000 "
         "chiffrements." % (LV == 1).sum(), fontsize=6, color="0.4")
sauver(fig, "escalier_v13.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.8 : Figure 6.4 — synthèse : canaux auxiliaires et redondance pilotée (hors ligne) ====
C = json.load(open(os.path.join(TR, "comparaison_sca_v12.json")))
EG = [c for c in C if c["surcout_pct"] > 40]                       # conditions à coût égal (≈ +43 %)
LAB = {"délais aléatoires": "délais\naléatoires", "opérations factices (type dual-rail)": "opérations\nfactices",
       "bruit ALU": "bruit\narithm.", "notre niveau 3 statique (ParEGO), s0": "niveau 3\n(camp. 1)",
       "notre niveau 3 statique (ParEGO), s1": "niveau 3\n(camp. 2)",
       "NOTRE MODÈLE déployé (arbre v12)": "modèle\ndéployé"}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.6), gridspec_kw=dict(width_ratios=[1.3, 1]))
x = np.arange(len(EG))
a1.bar(x, [c["effort_p10_8000"] for c in EG], width=0.6,
       color=[NOTRE if c["type"] == "nôtre" else CLASSIQUE for c in EG])
for xi, c in zip(x, EG):
    a1.text(xi, c["effort_p10_8000"] + 2, "%.0f" % c["effort_p10_8000"], ha="center", fontsize=6)
a1.axhline(32, color="0.4", lw=0.7, ls="--")
a1.set_xticks(x); a1.set_xticklabels([LAB[c["contre_mesure"]] for c in EG])
a1.set_ylabel("effort restant à 8 000 traces (bits)"); a1.set_ylim(0, 105)
a1.set_title("(a) Canaux auxiliaires, à coût égal (+43 %)", loc="left")
# Redondance : valeurs mesurées reprises des tableaux 6.3 et 6.4 du mémoire (cellules 12.5 et 12.6)
modes = ["aucune", "permanente", "pilotée\npar l'arbre"]
actif, cout_normal = [0, 100, 94], [0, 74, 0]
xx = np.arange(3)
a2.bar(xx - 0.18, actif, width=0.36, color="0.75", label="active sous attaque (%)")
a2.bar(xx + 0.18, cout_normal, width=0.36, color="0.3", label="surcoût en usage normal (%)")
for xi, (a, c) in enumerate(zip(actif, cout_normal)):
    a2.text(xi - 0.18, a + 2, "%d" % a, ha="center", fontsize=6); a2.text(xi + 0.18, c + 2, "%d" % c, ha="center", fontsize=6)
a2.set_xticks(xx); a2.set_xticklabels(modes); a2.set_ylim(0, 150)
a2.set_title("(b) Fautes : redondance pilotée", loc="left"); a2.legend(loc="upper left")
fig.tight_layout()
fig.text(0.0, -0.08, "(b) Usage normal sans redémarrage ; avec un redémarrage tous les 100 chiffrements, la redondance "
         "pilotée coûte +8,6 %.", fontsize=6, color="0.4")
sauver(fig, "synthese_v12.pdf"); plt.show()

In [ ]:
# ==== Cellule 16.9 : Figure §6.4 — entropie de la durée et sécurité (hors ligne, ≈ 20 s) ====
def H_duree(t):
    """Entropie de Shannon (bits) de la durée, sur 10 000 durées ; cases de 32 cycles si σ > 50 cycles."""
    t = np.asarray(t, np.int64)[:10000]
    w = 1 if t.std() < 50 else 32
    _, c = np.unique(t // w, return_counts=True); p = c / c.sum()
    return -(p * np.log2(p)).sum() + (len(c) - 1) / (2 * len(t) * np.log(2)) + np.log2(w)


EFF = {c["contre_mesure"]: c["effort_p10_8000"] for c in json.load(open(os.path.join(TR, "comparaison_sca_v12.json")))}
SRC = [("sans protection", "apprentissage/profil_sans_bruit_20k", "aucune protection"),
       ("opérations factices", "validation/classiques/operations_factices_20k", "opérations factices (type dual-rail)"),
       ("bruit arithmétique", "validation/classiques/bruit_alu_20k", "bruit ALU"),
       ("mélange seul", "apprentissage/profil_melange_30k", "mélange seul (shuffling)"),
       ("délais aléatoires", "validation/classiques/delais_aleatoires_20k", "délais aléatoires"),
       ("niveau 1", V3 % 100, None), ("niveau 2", V3 % 200, None),
       ("niveau 3 (camp. 1)", V3 % 536, "notre niveau 3 statique (ParEGO), s0"),
       ("niveau 3 (camp. 2)", "validation/s1/" + (V3 % 536).split("/", 1)[1], "notre niveau 3 statique (ParEGO), s1"),
       ("modèle v12", "validation/politique_v12_20k", "NOTRE MODÈLE déployé (arbre v12)")]
rows = [(n, H_duree(np.load(os.path.join(TR, rel, "trig.npy"))), EFF.get(k)) for n, rel, k in SRC]
rows.append(("modèle v13", H_duree(json.load(open(os.path.join(TR, "escalier_v13.json")))["trig"]), None))
for n, h, e in rows:
    print("%-20s entropie de la durée %5.1f bits | effort à 8 000 traces : %s" % (n, h, "—" if e is None else "%.1f bits" % e))
ours = lambda n: n.startswith(("niveau", "modèle"))
rs = sorted(rows, key=lambda r: r[1])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.9), gridspec_kw=dict(width_ratios=[1.05, 1]))
y = np.arange(len(rs)); cc = [NOTRE if ours(n) else CLASSIQUE for n, _, _ in rs]
a1.hlines(y, 0, [h for _, h, _ in rs], color=cc, lw=1.1); a1.scatter([h for _, h, _ in rs], y, color=cc, s=18, zorder=3)
for yi, (_, h, _) in zip(y, rs):
    a1.text(h + 0.3, yi, "%.1f" % h, va="center", fontsize=6)
a1.set_yticks(y); a1.set_yticklabels([n for n, _, _ in rs]); a1.set_xlim(0, 16.5)
a1.set_xlabel("entropie de la durée (bits)"); a1.set_title("(a) La durée varie avec l'amplitude des délais", loc="left")
for n, h, e in rows:
    if e is not None:
        a2.scatter(h, e, color=NOTRE if ours(n) else CLASSIQUE, s=22, zorder=3)
        if n in ("délais aléatoires", "niveau 3 (camp. 1)", "niveau 3 (camp. 2)", "modèle v12", "mélange seul"):
            a2.text(h - 0.35 if n != "mélange seul" else h + 0.35, e, n, ha="right" if n != "mélange seul" else "left",
                    va="center", fontsize=7)
a2.text(0.3, 14, "sans protection, opérations\nfactices, bruit arithmétique", fontsize=6.5, va="bottom")
a2.axhline(32, color="0.75", lw=0.7, ls="--")
a2.set_xlim(-0.5, 16.5); a2.set_ylim(-5, 100)
a2.set_xlabel("entropie de la durée (bits)"); a2.set_ylabel("effort restant à 8 000 traces (bits)")
a2.set_title("(b) L'entropie ne prédit pas la sécurité", loc="left")
fig.tight_layout()
fig.text(0.0, -0.06, "Entropie de Shannon sur 10 000 durées (cases de 32 cycles si σ > 50 cycles, correction de "
         "Miller-Madow).", fontsize=6, color="0.4")
sauver(fig, "entropie_durees.pdf"); plt.show()